<a href="https://colab.research.google.com/github/S3B4-11/sprintreview3/blob/main/Prediccion_Mundial_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================
# PASO 1: Cargar datos
# ============================

import pandas as pd
import numpy as np

# Base histórica de partidos internacionales
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

df = pd.read_csv(url)

# Ver las primeras filas
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [ ]:
# Ver información general de la base
print("Cantidad de partidos:", len(df))
print("Columnas disponibles:")
print(df.columns)

df.info()

Cantidad de partidos: 49520
Columnas disponibles:
Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49520 entries, 0 to 49519
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49520 non-null  object 
 1   home_team   49520 non-null  object 
 2   away_team   49520 non-null  object 
 3   home_score  49518 non-null  float64
 4   away_score  49518 non-null  float64
 5   tournament  49520 non-null  object 
 6   city        49520 non-null  object 
 7   country     49520 non-null  object 
 8   neutral     49520 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


In [ ]:
# ============================
# PASO 2: Limpiar la base
# ============================

# Convertir la columna date a formato fecha
df["date"] = pd.to_datetime(df["date"])

# Eliminar partidos sin marcador
df = df.dropna(subset=["home_score", "away_score"]).copy()

# Convertir goles a número entero
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

# Filtrar partidos desde el año 2000
df = df[df["date"].dt.year >= 2000].copy()

# Crear columnas útiles
df["total_goals"] = df["home_score"] + df["away_score"]
df["goal_difference"] = df["home_score"] - df["away_score"]

# Crear resultado del partido
# 1 = gana local, 0 = empate, -1 = gana visita
df["result"] = np.where(
    df["home_score"] > df["away_score"], 1,
    np.where(df["home_score"] < df["away_score"], -1, 0)
)

# Revisar resultado de limpieza
print("Partidos desde 2000:", len(df))
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

df.head()

Partidos desde 2000: 25456
Fecha mínima: 2000-01-04 00:00:00
Fecha máxima: 2026-07-15 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,total_goals,goal_difference,result
24062,2000-01-04,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False,3,1,1
24063,2000-01-07,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False,7,7,1
24064,2000-01-08,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False,0,0,0
24065,2000-01-09,Burkina Faso,Gabon,1,1,Friendly,Ouagadougou,Burkina Faso,False,2,0,0
24066,2000-01-09,Guatemala,Armenia,1,1,Friendly,Los Angeles,United States,True,2,0,0


In [ ]:
# ============================
# Revisión rápida
# ============================

print("Promedio de goles por partido:", round(df["total_goals"].mean(), 2))
print("Promedio goles local:", round(df["home_score"].mean(), 2))
print("Promedio goles visita:", round(df["away_score"].mean(), 2))

print("\nDistribución de resultados:")
print(df["result"].value_counts())

print("\nTorneos más frecuentes:")
print(df["tournament"].value_counts().head(10))

Promedio de goles por partido: 2.76
Promedio goles local: 1.64
Promedio goles visita: 1.11

Distribución de resultados:
result
 1    12254
-1     7274
 0     5928
Name: count, dtype: int64

Torneos más frecuentes:
tournament
Friendly                                8447
FIFA World Cup qualification            5840
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      658
AFC Asian Cup qualification              530
African Cup of Nations                   525
FIFA World Cup                           486
CONCACAF Nations League                  422
Gold Cup                                 359
Name: count, dtype: int64


In [ ]:
# ============================
# PASO 3: Filtrar partidos útiles para predecir Mundial
# ============================

# Torneos que nos sirven más para nivel de selecciones
# Incluye Mundial, clasificatorias, Euro, Copa América, Copa Africana,
# Copa Asiática, Gold Cup y Nations League.
torneos_utiles = [
    "FIFA World Cup",
    "UEFA Euro",
    "Copa América",
    "Copa America",
    "African Cup of Nations",
    "AFC Asian Cup",
    "Gold Cup",
    "UEFA Nations League",
    "CONCACAF Nations League",
    "Oceania Nations Cup"
]

# Crear filtro usando los nombres anteriores
patron_torneos = "|".join(torneos_utiles)

df_modelo = df[
    df["tournament"].str.contains(patron_torneos, case=False, na=False)
].copy()

# Sacamos amistosos por seguridad
df_modelo = df_modelo[
    ~df_modelo["tournament"].str.contains("Friendly", case=False, na=False)
].copy()

# Revisar cuántos partidos quedaron
print("Partidos originales desde 2000:", len(df))
print("Partidos útiles para el modelo:", len(df_modelo))

print("\nTorneos incluidos:")
print(df_modelo["tournament"].value_counts().head(20))

Partidos originales desde 2000: 25456
Partidos útiles para el modelo: 12816

Torneos incluidos:
tournament
FIFA World Cup qualification             5840
UEFA Euro qualification                  1532
African Cup of Nations qualification     1416
UEFA Nations League                       658
AFC Asian Cup qualification               530
African Cup of Nations                    525
FIFA World Cup                            486
CONCACAF Nations League                   422
Gold Cup                                  359
UEFA Euro                                 277
AFC Asian Cup                             256
Copa América                              248
Oceania Nations Cup                        99
Gold Cup qualification                     81
CONCACAF Nations League qualification      68
Oceania Nations Cup qualification          15
Copa América qualification                  4
Name: count, dtype: int64


In [ ]:
# ============================
# PASO 3.1: Crear peso del partido
# ============================

def asignar_peso_torneo(torneo):
    torneo = torneo.lower()

    if torneo == "fifa world cup":
        return 3.0
    elif "fifa world cup qualification" in torneo:
        return 2.3
    elif "uefa euro" in torneo or "copa américa" in torneo or "copa america" in torneo:
        return 2.0
    elif "african cup of nations" in torneo or "afc asian cup" in torneo or "gold cup" in torneo:
        return 1.8
    elif "nations league" in torneo:
        return 1.5
    else:
        return 1.0

df_modelo["peso_torneo"] = df_modelo["tournament"].apply(asignar_peso_torneo)

# Peso por año: partidos más recientes importan más
df_modelo["anio"] = df_modelo["date"].dt.year
df_modelo["peso_recencia"] = 1 + ((df_modelo["anio"] - 2000) / (df_modelo["anio"].max() - 2000))

# Peso final
df_modelo["peso_final"] = df_modelo["peso_torneo"] * df_modelo["peso_recencia"]

df_modelo[["date", "home_team", "away_team", "home_score", "away_score", "tournament", "peso_final"]].head()

,date,home_team,away_team,home_score,away_score,tournament,peso_final
24082,2000-01-22,Ghana,Cameroon,1,1,African Cup of Nations,1.8
24083,2000-01-23,China,Philippines,8,0,AFC Asian Cup qualification,1.8
24084,2000-01-23,Egypt,Zambia,2,0,African Cup of Nations,1.8
24086,2000-01-23,Nigeria,Tunisia,4,2,African Cup of Nations,1.8
24087,2000-01-23,South Africa,Gabon,3,1,African Cup of Nations,1.8


In [ ]:
# ============================
# Revisión del filtro
# ============================

print("Promedio goles por partido en base filtrada:", round(df_modelo["total_goals"].mean(), 2))
print("Promedio goles local:", round(df_modelo["home_score"].mean(), 2))
print("Promedio goles visita:", round(df_modelo["away_score"].mean(), 2))

print("\nCantidad de partidos por año:")
print(df_modelo["anio"].value_counts().sort_index().tail(10))

print("\nPartidos de Mundial disponibles:")
print(len(df_modelo[df_modelo["tournament"] == "FIFA World Cup"]))

Promedio goles por partido en base filtrada: 2.77
Promedio goles local: 1.66
Promedio goles visita: 1.11

Cantidad de partidos por año:
anio
2017    455
2018    357
2019    749
2020    243
2021    788
2022    533
2023    635
2024    893
2025    664
2026    134
Name: count, dtype: int64

Partidos de Mundial disponibles:
486


In [ ]:
# ============================
# PASO 4: Crear estadísticas recientes por selección
# ============================

from collections import defaultdict

# Ordenar partidos por fecha
df_modelo = df_modelo.sort_values("date").reset_index(drop=True)

# Promedios globales para usar cuando un equipo tenga pocos partidos previos
prom_goles_equipo = (df_modelo["home_score"].sum() + df_modelo["away_score"].sum()) / (len(df_modelo) * 2)
prom_puntos_equipo = 1.33

historial = defaultdict(list)
filas_modelo = []

def calcular_stats(partidos_previos, n=10):
    """
    Calcula rendimiento reciente de un equipo usando sus últimos n partidos.
    """
    if len(partidos_previos) == 0:
        return {
            "gf_prom": prom_goles_equipo,
            "ga_prom": prom_goles_equipo,
            "dif_prom": 0,
            "pts_prom": prom_puntos_equipo,
            "victorias_pct": 0.33,
            "empates_pct": 0.25,
            "derrotas_pct": 0.42,
            "partidos_previos": 0
        }

    ultimos = partidos_previos[-n:]

    gf = np.mean([p["gf"] for p in ultimos])
    ga = np.mean([p["ga"] for p in ultimos])
    dif = np.mean([p["gf"] - p["ga"] for p in ultimos])
    pts = np.mean([p["pts"] for p in ultimos])

    victorias = np.mean([1 if p["pts"] == 3 else 0 for p in ultimos])
    empates = np.mean([1 if p["pts"] == 1 else 0 for p in ultimos])
    derrotas = np.mean([1 if p["pts"] == 0 else 0 for p in ultimos])

    return {
        "gf_prom": gf,
        "ga_prom": ga,
        "dif_prom": dif,
        "pts_prom": pts,
        "victorias_pct": victorias,
        "empates_pct": empates,
        "derrotas_pct": derrotas,
        "partidos_previos": len(partidos_previos)
    }

for _, partido in df_modelo.iterrows():
    local = partido["home_team"]
    visita = partido["away_team"]

    # Estadísticas ANTES del partido
    stats_local = calcular_stats(historial[local], n=10)
    stats_visita = calcular_stats(historial[visita], n=10)

    fila = {
        "date": partido["date"],
        "home_team": local,
        "away_team": visita,
        "tournament": partido["tournament"],
        "neutral": int(partido["neutral"]),
        "anio": partido["anio"],
        "peso_final": partido["peso_final"],

        # Estadísticas local
        "home_gf_prom": stats_local["gf_prom"],
        "home_ga_prom": stats_local["ga_prom"],
        "home_dif_prom": stats_local["dif_prom"],
        "home_pts_prom": stats_local["pts_prom"],
        "home_victorias_pct": stats_local["victorias_pct"],
        "home_empates_pct": stats_local["empates_pct"],
        "home_derrotas_pct": stats_local["derrotas_pct"],
        "home_partidos_previos": stats_local["partidos_previos"],

        # Estadísticas visita
        "away_gf_prom": stats_visita["gf_prom"],
        "away_ga_prom": stats_visita["ga_prom"],
        "away_dif_prom": stats_visita["dif_prom"],
        "away_pts_prom": stats_visita["pts_prom"],
        "away_victorias_pct": stats_visita["victorias_pct"],
        "away_empates_pct": stats_visita["empates_pct"],
        "away_derrotas_pct": stats_visita["derrotas_pct"],
        "away_partidos_previos": stats_visita["partidos_previos"],

        # Diferencias entre equipos
        "dif_gf_prom": stats_local["gf_prom"] - stats_visita["gf_prom"],
        "dif_ga_prom": stats_local["ga_prom"] - stats_visita["ga_prom"],
        "dif_pts_prom": stats_local["pts_prom"] - stats_visita["pts_prom"],
        "dif_dif_prom": stats_local["dif_prom"] - stats_visita["dif_prom"],

        # Objetivos reales a predecir
        "home_score": partido["home_score"],
        "away_score": partido["away_score"],
        "total_goals": partido["total_goals"],
        "goal_difference": partido["goal_difference"],
        "result": partido["result"]
    }

    filas_modelo.append(fila)

    # Ahora sí agregamos el partido al historial, después de usar las stats previas

    # Puntos local
    if partido["home_score"] > partido["away_score"]:
        pts_local = 3
        pts_visita = 0
    elif partido["home_score"] < partido["away_score"]:
        pts_local = 0
        pts_visita = 3
    else:
        pts_local = 1
        pts_visita = 1

    historial[local].append({
        "gf": partido["home_score"],
        "ga": partido["away_score"],
        "pts": pts_local
    })

    historial[visita].append({
        "gf": partido["away_score"],
        "ga": partido["home_score"],
        "pts": pts_visita
    })

# Crear dataframe final para modelo
df_ml = pd.DataFrame(filas_modelo)

print("Base para machine learning creada:", df_ml.shape)
df_ml.head()

Base para machine learning creada: (12816, 32)


,date,home_team,away_team,tournament,neutral,anio,peso_final,home_gf_prom,home_ga_prom,home_dif_prom,...,away_partidos_previos,dif_gf_prom,dif_ga_prom,dif_pts_prom,dif_dif_prom,home_score,away_score,total_goals,goal_difference,result
0,2000-01-22,Ghana,Cameroon,African Cup of Nations,0,2000,1.8,1.382608,1.382608,0.0,...,0,0.0,0.0,0.0,0.0,1,1,2,0,0
1,2000-01-23,China,Philippines,AFC Asian Cup qualification,1,2000,1.8,1.382608,1.382608,0.0,...,0,0.0,0.0,0.0,0.0,8,0,8,8,1
2,2000-01-23,Egypt,Zambia,African Cup of Nations,1,2000,1.8,1.382608,1.382608,0.0,...,0,0.0,0.0,0.0,0.0,2,0,2,2,1
3,2000-01-23,Nigeria,Tunisia,African Cup of Nations,0,2000,1.8,1.382608,1.382608,0.0,...,0,0.0,0.0,0.0,0.0,4,2,6,2,1
4,2000-01-23,South Africa,Gabon,African Cup of Nations,1,2000,1.8,1.382608,1.382608,0.0,...,0,0.0,0.0,0.0,0.0,3,1,4,2,1


In [ ]:
# ============================
# Revisión de la base ML
# ============================

print("Filas:", df_ml.shape[0])
print("Columnas:", df_ml.shape[1])

print("\nColumnas creadas:")
print(df_ml.columns)

print("\nEjemplo de partidos recientes:")
df_ml.tail(10)[[
    "date", "home_team", "away_team",
    "home_gf_prom", "away_gf_prom",
    "home_pts_prom", "away_pts_prom",
    "home_score", "away_score"
]]

Filas: 12816
Columnas: 32

Columnas creadas:
Index(['date', 'home_team', 'away_team', 'tournament', 'neutral', 'anio',
       'peso_final', 'home_gf_prom', 'home_ga_prom', 'home_dif_prom',
       'home_pts_prom', 'home_victorias_pct', 'home_empates_pct',
       'home_derrotas_pct', 'home_partidos_previos', 'away_gf_prom',
       'away_ga_prom', 'away_dif_prom', 'away_pts_prom', 'away_victorias_pct',
       'away_empates_pct', 'away_derrotas_pct', 'away_partidos_previos',
       'dif_gf_prom', 'dif_ga_prom', 'dif_pts_prom', 'dif_dif_prom',
       'home_score', 'away_score', 'total_goals', 'goal_difference', 'result'],
      dtype='object')

Ejemplo de partidos recientes:


,date,home_team,away_team,home_gf_prom,away_gf_prom,home_pts_prom,away_pts_prom,home_score,away_score
12806,2026-07-06,Portugal,Spain,2.8,2.9,2.1,2.6,0,1
12807,2026-07-06,United States,Belgium,2.3,3.3,2.2,2.2,1,4
12808,2026-07-07,Argentina,Egypt,2.1,1.3,2.5,1.7,3,2
12809,2026-07-07,Switzerland,Colombia,2.3,1.8,2.4,1.9,0,0
12810,2026-07-09,France,Morocco,2.8,1.9,2.8,2.4,2,0
12811,2026-07-10,Spain,Belgium,2.7,3.1,2.6,2.2,2,1
12812,2026-07-11,Norway,England,3.7,2.7,2.7,2.8,1,2
12813,2026-07-11,Argentina,Switzerland,2.3,1.9,2.5,2.2,3,1
12814,2026-07-14,France,Spain,2.8,2.3,2.8,2.6,0,2
12815,2026-07-15,England,Argentina,2.7,2.2,2.8,2.5,1,2


In [ ]:
# ============================
# PASO 5: Crear ranking Elo propio por selección
# ============================

df_ml = df_ml.sort_values("date").reset_index(drop=True)

# Diccionario de Elo inicial
elo = defaultdict(lambda: 1500)

filas_elo = []

def resultado_real(goles_a, goles_b):
    if goles_a > goles_b:
        return 1
    elif goles_a < goles_b:
        return 0
    else:
        return 0.5

def probabilidad_elo(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

for _, partido in df_ml.iterrows():
    local = partido["home_team"]
    visita = partido["away_team"]

    elo_local_antes = elo[local]
    elo_visita_antes = elo[visita]

    prob_local = probabilidad_elo(elo_local_antes, elo_visita_antes)
    prob_visita = 1 - prob_local

    fila = partido.to_dict()
    fila["home_elo"] = elo_local_antes
    fila["away_elo"] = elo_visita_antes
    fila["dif_elo"] = elo_local_antes - elo_visita_antes
    fila["prob_home_elo"] = prob_local
    fila["prob_away_elo"] = prob_visita

    filas_elo.append(fila)

    # Actualizar Elo después del partido
    real_local = resultado_real(partido["home_score"], partido["away_score"])
    real_visita = 1 - real_local

    # K más alto para partidos más importantes
    k = 20 * partido["peso_final"]

    elo[local] = elo_local_antes + k * (real_local - prob_local)
    elo[visita] = elo_visita_antes + k * (real_visita - prob_visita)

df_ml = pd.DataFrame(filas_elo)

print("Base con Elo creada:", df_ml.shape)

df_ml.tail(10)[[
    "date", "home_team", "away_team",
    "home_elo", "away_elo", "dif_elo",
    "home_score", "away_score"
]]

Base con Elo creada: (12816, 37)


,date,home_team,away_team,home_elo,away_elo,dif_elo,home_score,away_score
12806,2026-07-06,Portugal,Spain,1953.592017,2078.414034,-124.822017,0,1
12807,2026-07-06,United States,Belgium,1913.311197,1893.683590,19.627607,1,4
12808,2026-07-07,Argentina,Egypt,2036.497012,1861.627094,174.869918,3,2
12809,2026-07-07,Switzerland,Colombia,1918.735997,1962.426244,-43.690246,0,0
12810,2026-07-09,France,Morocco,2113.446105,2007.995191,105.450914,2,0
12811,2026-07-10,Spain,Belgium,2117.739959,1957.069557,160.670402,2,1
12812,2026-07-11,Norway,England,2023.089581,2088.315771,-65.226190,1,2
12813,2026-07-11,Argentina,Switzerland,2068.613747,1926.241515,142.372232,3,1
12814,2026-07-14,France,Spain,2155.774701,2151.815390,3.959311,0,2
12815,2026-07-15,England,Argentina,2137.182102,2105.316584,31.865518,1,2


In [ ]:
# ============================
# Ranking Elo actual estimado
# ============================

ranking_elo = pd.DataFrame({
    "team": list(elo.keys()),
    "elo": list(elo.values())
}).sort_values("elo", ascending=False)

ranking_elo.head(30)

,team,elo
140,Spain,2212.499109
70,Argentina,2170.804185
128,France,2095.090982
137,England,2071.694500
33,Mexico,2003.362773
141,Norway,1974.223249
18,Morocco,1965.666594
24,Colombia,1954.920726
126,Belgium,1922.994126
130,Netherlands,1917.694295


In [ ]:
# ============================
# PASO 6: Entrenar modelo de goles
# ============================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math

# Variables que usará el modelo
features = [
    "neutral",
    "home_gf_prom", "home_ga_prom", "home_dif_prom", "home_pts_prom",
    "home_victorias_pct", "home_empates_pct", "home_derrotas_pct",
    "home_partidos_previos",
    "away_gf_prom", "away_ga_prom", "away_dif_prom", "away_pts_prom",
    "away_victorias_pct", "away_empates_pct", "away_derrotas_pct",
    "away_partidos_previos",
    "dif_gf_prom", "dif_ga_prom", "dif_pts_prom", "dif_dif_prom",
    "home_elo", "away_elo", "dif_elo",
    "prob_home_elo", "prob_away_elo"
]

# Separación temporal: entrenamos hasta 2023 y probamos desde 2024
train = df_ml[df_ml["date"].dt.year <= 2023].copy()
test = df_ml[df_ml["date"].dt.year >= 2024].copy()

X_train = train[features]
X_test = test[features]

y_home_train = train["home_score"]
y_home_test = test["home_score"]

y_away_train = train["away_score"]
y_away_test = test["away_score"]

peso_train = train["peso_final"]

# Modelo para goles local
modelo_home = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=8,
    random_state=42
)

# Modelo para goles visita
modelo_away = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=8,
    random_state=42
)

modelo_home.fit(X_train, y_home_train, sample_weight=peso_train)
modelo_away.fit(X_train, y_away_train, sample_weight=peso_train)

# Predicciones en test
pred_home = modelo_home.predict(X_test)
pred_away = modelo_away.predict(X_test)

# Evaluación
mae_home = mean_absolute_error(y_home_test, pred_home)
mae_away = mean_absolute_error(y_away_test, pred_away)

rmse_home = math.sqrt(mean_squared_error(y_home_test, pred_home))
rmse_away = math.sqrt(mean_squared_error(y_away_test, pred_away))

print("Evaluación modelo goles local:")
print("MAE:", round(mae_home, 3))
print("RMSE:", round(rmse_home, 3))

print("\nEvaluación modelo goles visita:")
print("MAE:", round(mae_away, 3))
print("RMSE:", round(rmse_away, 3))

Evaluación modelo goles local:
MAE: 1.032
RMSE: 1.347

Evaluación modelo goles visita:
MAE: 0.845
RMSE: 1.108


In [ ]:
# ============================
# PASO 7: Comparar varios modelos
# ============================

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math
import pandas as pd
import numpy as np

modelos = {
    "RandomForest_base": RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=8,
        random_state=42
    ),

    "RandomForest_mas_profundo": RandomForestRegressor(
        n_estimators=800,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.03,
        max_leaf_nodes=20,
        random_state=42
    ),

    "PoissonRegressor": make_pipeline(
        StandardScaler(),
        PoissonRegressor(alpha=0.1, max_iter=1000)
    )
}

resultados = []

for nombre, modelo in modelos.items():

    # Modelo goles local
    modelo_h = modelo
    modelo_h.fit(X_train, y_home_train)
    pred_h = modelo_h.predict(X_test)
    pred_h = np.clip(pred_h, 0, 8)

    mae_h = mean_absolute_error(y_home_test, pred_h)
    rmse_h = math.sqrt(mean_squared_error(y_home_test, pred_h))

    # Modelo goles visita
    # Se crea otro modelo igual para evitar que se mezcle el entrenamiento
    import copy
    modelo_a = copy.deepcopy(modelo)
    modelo_a.fit(X_train, y_away_train)
    pred_a = modelo_a.predict(X_test)
    pred_a = np.clip(pred_a, 0, 8)

    mae_a = mean_absolute_error(y_away_test, pred_a)
    rmse_a = math.sqrt(mean_squared_error(y_away_test, pred_a))

    resultados.append({
        "modelo": nombre,
        "MAE_local": round(mae_h, 3),
        "RMSE_local": round(rmse_h, 3),
        "MAE_visita": round(mae_a, 3),
        "RMSE_visita": round(rmse_a, 3),
        "MAE_promedio": round((mae_h + mae_a) / 2, 3)
    })

tabla_resultados = pd.DataFrame(resultados).sort_values("MAE_promedio")
tabla_resultados

,modelo,MAE_local,RMSE_local,MAE_visita,RMSE_visita,MAE_promedio
4,PoissonRegressor,1.017,1.356,0.842,1.102,0.930
2,GradientBoosting,1.029,1.353,0.842,1.101,0.936
0,RandomForest_base,1.030,1.345,0.846,1.107,0.938
3,HistGradientBoosting,1.038,1.356,0.840,1.097,0.939
1,RandomForest_mas_profundo,1.035,1.346,0.848,1.106,0.942


In [ ]:
# ============================
# PASO 9: Modelo final con Poisson
# ============================

from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import math
from scipy.stats import poisson

# Usaremos toda la base disponible para entrenar el modelo final
X_final = df_ml[features]
y_home_final = df_ml["home_score"]
y_away_final = df_ml["away_score"]

modelo_final_home = make_pipeline(
    StandardScaler(),
    PoissonRegressor(alpha=0.1, max_iter=1000)
)

modelo_final_away = make_pipeline(
    StandardScaler(),
    PoissonRegressor(alpha=0.1, max_iter=1000)
)

modelo_final_home.fit(X_final, y_home_final)
modelo_final_away.fit(X_final, y_away_final)

print("Modelos finales entrenados correctamente.")

Modelos finales entrenados correctamente.


In [ ]:
# ============================
# PASO 10: Función para predecir partidos
# ============================

def crear_features_partido(equipo_local, equipo_visita, neutral=1):
    """
    Crea las variables necesarias para predecir un partido nuevo.
    Usa las estadísticas recientes y Elo actual de cada selección.
    """

    stats_local = calcular_stats(historial[equipo_local], n=10)
    stats_visita = calcular_stats(historial[equipo_visita], n=10)

    home_elo = elo[equipo_local]
    away_elo = elo[equipo_visita]
    dif_elo = home_elo - away_elo

    prob_home = probabilidad_elo(home_elo, away_elo)
    prob_away = 1 - prob_home

    fila = {
        "neutral": int(neutral),

        "home_gf_prom": stats_local["gf_prom"],
        "home_ga_prom": stats_local["ga_prom"],
        "home_dif_prom": stats_local["dif_prom"],
        "home_pts_prom": stats_local["pts_prom"],
        "home_victorias_pct": stats_local["victorias_pct"],
        "home_empates_pct": stats_local["empates_pct"],
        "home_derrotas_pct": stats_local["derrotas_pct"],
        "home_partidos_previos": stats_local["partidos_previos"],

        "away_gf_prom": stats_visita["gf_prom"],
        "away_ga_prom": stats_visita["ga_prom"],
        "away_dif_prom": stats_visita["dif_prom"],
        "away_pts_prom": stats_visita["pts_prom"],
        "away_victorias_pct": stats_visita["victorias_pct"],
        "away_empates_pct": stats_visita["empates_pct"],
        "away_derrotas_pct": stats_visita["derrotas_pct"],
        "away_partidos_previos": stats_visita["partidos_previos"],

        "dif_gf_prom": stats_local["gf_prom"] - stats_visita["gf_prom"],
        "dif_ga_prom": stats_local["ga_prom"] - stats_visita["ga_prom"],
        "dif_pts_prom": stats_local["pts_prom"] - stats_visita["pts_prom"],
        "dif_dif_prom": stats_local["dif_prom"] - stats_visita["dif_prom"],

        "home_elo": home_elo,
        "away_elo": away_elo,
        "dif_elo": dif_elo,
        "prob_home_elo": prob_home,
        "prob_away_elo": prob_away
    }

    return pd.DataFrame([fila])[features]


def calcular_probabilidades_marcador(goles_home_esp, goles_away_esp, max_goles=7):
    """
    Calcula probabilidades de victoria, empate y derrota usando distribución Poisson.
    """

    prob_home_win = 0
    prob_draw = 0
    prob_away_win = 0

    marcador_mas_probable = None
    prob_marcador_mas_probable = 0

    for gh in range(max_goles + 1):
        for ga in range(max_goles + 1):
            prob = poisson.pmf(gh, goles_home_esp) * poisson.pmf(ga, goles_away_esp)

            if gh > ga:
                prob_home_win += prob
            elif gh == ga:
                prob_draw += prob
            else:
                prob_away_win += prob

            if prob > prob_marcador_mas_probable:
                prob_marcador_mas_probable = prob
                marcador_mas_probable = (gh, ga)

    return prob_home_win, prob_draw, prob_away_win, marcador_mas_probable, prob_marcador_mas_probable


def predecir_partido(equipo_local, equipo_visita, neutral=1):
    """
    Predice goles esperados, probabilidades y marcador más probable.
    """

    X_partido = crear_features_partido(equipo_local, equipo_visita, neutral)

    goles_home_esp = modelo_final_home.predict(X_partido)[0]
    goles_away_esp = modelo_final_away.predict(X_partido)[0]

    goles_home_esp = max(0, goles_home_esp)
    goles_away_esp = max(0, goles_away_esp)

    p_home, p_draw, p_away, marcador, p_marcador = calcular_probabilidades_marcador(
        goles_home_esp,
        goles_away_esp
    )

    resultado = pd.DataFrame([{
        "equipo_1": equipo_local,
        "equipo_2": equipo_visita,
        "goles_esperados_equipo_1": round(goles_home_esp, 2),
        "goles_esperados_equipo_2": round(goles_away_esp, 2),
        "prob_gana_equipo_1": round(p_home * 100, 1),
        "prob_empate": round(p_draw * 100, 1),
        "prob_gana_equipo_2": round(p_away * 100, 1),
        "marcador_mas_probable": f"{marcador[0]} - {marcador[1]}",
        "prob_marcador": round(p_marcador * 100, 1)
    }])

    return resultado

In [ ]:
predecir_partido("Mexico", "Ecuador")

,equipo_1,equipo_2,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,marcador_mas_probable,prob_marcador
0,Mexico,Ecuador,1.57,0.67,59.1,25.2,15.7,1 - 0,16.7


In [ ]:
predecir_partido("Argentina", "Cape Verde")

,equipo_1,equipo_2,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,marcador_mas_probable,prob_marcador
0,Argentina,Cape Verde,2.81,0.46,84.8,10.5,3.8,2 - 0,15.0


In [ ]:
predecir_partido("England", "DR Congo")

,equipo_1,equipo_2,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,marcador_mas_probable,prob_marcador
0,England,DR Congo,1.92,0.62,68.3,20.4,11.2,1 - 0,15.1


In [ ]:
# ============================
# PASO 12: Predecir varios partidos
# ============================

partidos_mundial = [
    ("Mexico", "Ecuador"),
    ("Argentina", "Cape Verde"),
    ("England", "DR Congo"),
    ("Belgium", "Senegal"),
    ("United States", "Bosnia and Herzegovina"),
    ("Spain", "Austria"),
    ("Portugal", "Croatia"),
    ("Switzerland", "Algeria"),
    ("Australia", "Egypt"),
    ("Colombia", "Ghana"),
    ("Brazil", "Norway"),
    ("France", "Paraguay"),
    ("Canada", "Morocco")
]

def predecir_varios_partidos(lista_partidos):
    resultados = []

    for equipo_1, equipo_2 in lista_partidos:
        pred = predecir_partido(equipo_1, equipo_2, neutral=1)
        resultados.append(pred)

    tabla = pd.concat(resultados, ignore_index=True)

    # Agregar una columna con el favorito según probabilidad
    tabla["favorito_modelo"] = tabla.apply(
        lambda x: x["equipo_1"] if x["prob_gana_equipo_1"] > x["prob_gana_equipo_2"] else x["equipo_2"],
        axis=1
    )

    tabla["prob_favorito"] = tabla.apply(
        lambda x: max(x["prob_gana_equipo_1"], x["prob_gana_equipo_2"]),
        axis=1
    )

    return tabla.sort_values("prob_favorito", ascending=False)

predicciones_mundial = predecir_varios_partidos(partidos_mundial)

predicciones_mundial

,equipo_1,equipo_2,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,marcador_mas_probable,prob_marcador,favorito_modelo,prob_favorito
5,Spain,Austria,2.87,0.44,85.6,10.0,3.5,2 - 0,15.0,Spain,85.6
1,Argentina,Cape Verde,2.81,0.46,84.8,10.5,3.8,2 - 0,15.0,Argentina,84.8
11,France,Paraguay,1.94,0.62,68.7,20.2,11.0,1 - 0,15.0,France,68.7
2,England,DR Congo,1.92,0.62,68.3,20.4,11.2,1 - 0,15.1,England,68.3
4,United States,Bosnia and Herzegovina,1.82,0.71,63.8,22.1,14.0,1 - 0,14.5,United States,63.8
0,Mexico,Ecuador,1.57,0.67,59.1,25.2,15.7,1 - 0,16.7,Mexico,59.1
9,Colombia,Ghana,1.50,0.67,57.3,26.1,16.5,1 - 0,17.2,Colombia,57.3
3,Belgium,Senegal,1.63,0.79,57.2,24.6,18.1,1 - 0,14.4,Belgium,57.2
7,Switzerland,Algeria,1.51,0.74,55.5,26.1,18.4,1 - 0,15.9,Switzerland,55.5
12,Canada,Morocco,0.81,1.57,19.2,25.3,55.5,0 - 1,14.5,Morocco,55.5


In [ ]:
# ============================
# PASO 13: Tabla final más limpia
# ============================

tabla_final = predicciones_mundial[[
    "equipo_1",
    "equipo_2",
    "goles_esperados_equipo_1",
    "goles_esperados_equipo_2",
    "prob_gana_equipo_1",
    "prob_empate",
    "prob_gana_equipo_2",
    "marcador_mas_probable",
    "favorito_modelo",
    "prob_favorito"
]]

tabla_final

,equipo_1,equipo_2,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,marcador_mas_probable,favorito_modelo,prob_favorito
5,Spain,Austria,2.87,0.44,85.6,10.0,3.5,2 - 0,Spain,85.6
1,Argentina,Cape Verde,2.81,0.46,84.8,10.5,3.8,2 - 0,Argentina,84.8
11,France,Paraguay,1.94,0.62,68.7,20.2,11.0,1 - 0,France,68.7
2,England,DR Congo,1.92,0.62,68.3,20.4,11.2,1 - 0,England,68.3
4,United States,Bosnia and Herzegovina,1.82,0.71,63.8,22.1,14.0,1 - 0,United States,63.8
0,Mexico,Ecuador,1.57,0.67,59.1,25.2,15.7,1 - 0,Mexico,59.1
9,Colombia,Ghana,1.50,0.67,57.3,26.1,16.5,1 - 0,Colombia,57.3
3,Belgium,Senegal,1.63,0.79,57.2,24.6,18.1,1 - 0,Belgium,57.2
7,Switzerland,Algeria,1.51,0.74,55.5,26.1,18.4,1 - 0,Switzerland,55.5
12,Canada,Morocco,0.81,1.57,19.2,25.3,55.5,0 - 1,Morocco,55.5


In [ ]:
# ============================
# PASO 15: Cargar competiciones disponibles de StatsBomb Open Data
# ============================

import pandas as pd
import requests

url_competiciones = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json"

competiciones = requests.get(url_competiciones).json()
df_competiciones = pd.DataFrame(competiciones)

df_competiciones[[
    "competition_id",
    "season_id",
    "country_name",
    "competition_name",
    "season_name"
]].head(30)

,competition_id,season_id,country_name,competition_name,season_name
0,9,281,Germany,1. Bundesliga,2023/2024
1,9,27,Germany,1. Bundesliga,2015/2016
2,1267,107,Africa,African Cup of Nations,2023
3,16,4,Europe,Champions League,2018/2019
4,16,1,Europe,Champions League,2017/2018
5,16,2,Europe,Champions League,2016/2017
6,16,27,Europe,Champions League,2015/2016
7,16,26,Europe,Champions League,2014/2015
8,16,25,Europe,Champions League,2013/2014
9,16,24,Europe,Champions League,2012/2013


In [ ]:
# Buscar competiciones relacionadas con World Cup o selecciones
df_competiciones[
    df_competiciones["competition_name"].str.contains("World Cup|Euro|Copa", case=False, na=False)
][[
    "competition_id",
    "season_id",
    "country_name",
    "competition_name",
    "season_name"
]]

,competition_id,season_id,country_name,competition_name,season_name
21,223,282,South America,Copa America,2024
22,87,84,Spain,Copa del Rey,1983/1984
23,87,268,Spain,Copa del Rey,1982/1983
24,87,279,Spain,Copa del Rey,1977/1978
29,1470,274,International,FIFA U20 World Cup,1979
30,43,106,International,FIFA World Cup,2022
31,43,3,International,FIFA World Cup,2018
32,43,55,International,FIFA World Cup,1990
33,43,54,International,FIFA World Cup,1986
34,43,51,International,FIFA World Cup,1974


In [ ]:
# ============================
# PASO 16: Cargar partidos del Mundial 2022
# ============================

import pandas as pd
import requests
import numpy as np

competition_id = 43
season_id = 106

url_matches = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/{competition_id}/{season_id}.json"

matches = requests.get(url_matches).json()
df_matches = pd.json_normalize(matches)

columnas_partidos = [
    "match_id",
    "match_date",
    "home_team.home_team_name",
    "away_team.away_team_name",
    "home_score",
    "away_score",
    "competition.competition_name",
    "season.season_name"
]

df_matches[columnas_partidos].head(10)

,match_id,match_date,home_team.home_team_name,away_team.away_team_name,home_score,away_score,competition.competition_name,season.season_name
0,3857276,2022-12-01,Canada,Morocco,1,2,FIFA World Cup,2022
1,3857271,2022-11-21,England,Iran,6,2,FIFA World Cup,2022
2,3857296,2022-12-01,Croatia,Belgium,0,0,FIFA World Cup,2022
3,3857274,2022-11-25,Netherlands,Ecuador,1,1,FIFA World Cup,2022
4,3857255,2022-12-01,Japan,Spain,2,1,FIFA World Cup,2022
5,3857272,2022-11-25,England,United States,0,0,FIFA World Cup,2022
6,3857278,2022-11-29,Iran,United States,0,1,FIFA World Cup,2022
7,3857277,2022-11-23,Morocco,Croatia,0,0,FIFA World Cup,2022
8,3857273,2022-11-25,Wales,Iran,0,2,FIFA World Cup,2022
9,3857275,2022-11-30,Tunisia,France,1,0,FIFA World Cup,2022


In [ ]:
# ============================
# PASO 17: Cargar eventos de un partido
# ============================

match_id_ejemplo = df_matches.iloc[0]["match_id"]

url_events = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id_ejemplo}.json"

events = requests.get(url_events).json()
df_events = pd.json_normalize(events)

print("Match ID:", match_id_ejemplo)
print("Cantidad de eventos:", len(df_events))

df_events[["team.name", "type.name"]].head(20)

Match ID: 3857276
Cantidad de eventos: 3388


,team.name,type.name
0,Canada,Starting XI
1,Morocco,Starting XI
2,Morocco,Half Start
3,Canada,Half Start
4,Morocco,Pass
5,Morocco,Ball Receipt*
6,Morocco,Carry
7,Morocco,Pass
8,Morocco,Ball Receipt*
9,Morocco,Carry


In [ ]:
# ============================
# PASO 18: Función para sacar estadísticas del partido
# ============================

def estadisticas_partido_statsbomb(match_id):
    """
    Extrae estadísticas de un partido usando eventos de StatsBomb.
    """

    url_events = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id}.json"
    events = requests.get(url_events).json()
    df_e = pd.json_normalize(events)

    equipos = df_e["team.name"].dropna().unique()
    resultados = []

    for equipo in equipos:
        data = df_e[df_e["team.name"] == equipo].copy()

        # Eventos principales
        pases = data[data["type.name"] == "Pass"]
        tiros = data[data["type.name"] == "Shot"]
        carries = data[data["type.name"] == "Carry"]
        presiones = data[data["type.name"] == "Pressure"]
        faltas = data[data["type.name"] == "Foul Committed"]

        # Pases completados: en StatsBomb, pass.outcome.name vacío suele indicar pase exitoso
        if "pass.outcome.name" in pases.columns:
            pases_completados = pases[pases["pass.outcome.name"].isna()]
        else:
            pases_completados = pases

        # Corners y saques de banda
        if "pass.type.name" in pases.columns:
            corners = pases[pases["pass.type.name"] == "Corner"]
            saques_banda = pases[pases["pass.type.name"] == "Throw-in"]
            tiros_libres = pases[pases["pass.type.name"] == "Free Kick"]
        else:
            corners = pd.DataFrame()
            saques_banda = pd.DataFrame()
            tiros_libres = pd.DataFrame()

        # Tiros al arco aproximados
        if "shot.outcome.name" in tiros.columns:
            tiros_al_arco = tiros[
                tiros["shot.outcome.name"].isin(["Goal", "Saved", "Saved To Post"])
            ]
            goles = tiros[tiros["shot.outcome.name"] == "Goal"]
        else:
            tiros_al_arco = pd.DataFrame()
            goles = pd.DataFrame()

        # Tarjetas
        tarjetas_amarillas = 0
        tarjetas_rojas = 0

        if "foul_committed.card.name" in data.columns:
            tarjetas_amarillas += data["foul_committed.card.name"].fillna("").str.contains("Yellow", case=False).sum()
            tarjetas_rojas += data["foul_committed.card.name"].fillna("").str.contains("Red", case=False).sum()

        if "bad_behaviour.card.name" in data.columns:
            tarjetas_amarillas += data["bad_behaviour.card.name"].fillna("").str.contains("Yellow", case=False).sum()
            tarjetas_rojas += data["bad_behaviour.card.name"].fillna("").str.contains("Red", case=False).sum()

        resultados.append({
            "match_id": match_id,
            "equipo": equipo,
            "pases": len(pases),
            "pases_completados": len(pases_completados),
            "precision_pases_%": round((len(pases_completados) / len(pases)) * 100, 1) if len(pases) > 0 else 0,
            "tiros": len(tiros),
            "tiros_al_arco": len(tiros_al_arco),
            "goles_eventos": len(goles),
            "corners": len(corners),
            "saques_banda": len(saques_banda),
            "tiros_libres": len(tiros_libres),
            "carries": len(carries),
            "presiones": len(presiones),
            "faltas_cometidas": len(faltas),
            "tarjetas_amarillas": int(tarjetas_amarillas),
            "tarjetas_rojas": int(tarjetas_rojas)
        })

    return pd.DataFrame(resultados)

In [ ]:
estadisticas_partido_statsbomb(match_id_ejemplo)

,match_id,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,goles_eventos,corners,saques_banda,tiros_libres,carries,presiones,faltas_cometidas,tarjetas_amarillas,tarjetas_rojas
0,3857276,Canada,562,480,85.4,5,0,0,6,23,18,454,135,16,4,0
1,3857276,Morocco,393,310,78.9,7,2,2,2,20,17,328,154,15,0,0


In [ ]:
# ============================
# PASO 20: Estadísticas de todos los partidos del Mundial 2022
# ============================

todas_stats = []

for i, row in df_matches.iterrows():
    match_id = row["match_id"]

    stats = estadisticas_partido_statsbomb(match_id)

    stats["fecha"] = row["match_date"]
    stats["local"] = row["home_team.home_team_name"]
    stats["visita"] = row["away_team.away_team_name"]
    stats["goles_local"] = row["home_score"]
    stats["goles_visita"] = row["away_score"]

    todas_stats.append(stats)

    print(f"Procesado {i+1}/{len(df_matches)} - Match ID {match_id}")

df_stats_wc2022 = pd.concat(todas_stats, ignore_index=True)

df_stats_wc2022.head()

Procesado 1/64 - Match ID 3857276
Procesado 2/64 - Match ID 3857271
Procesado 3/64 - Match ID 3857296
Procesado 4/64 - Match ID 3857274
Procesado 5/64 - Match ID 3857255
Procesado 6/64 - Match ID 3857272
Procesado 7/64 - Match ID 3857278
Procesado 8/64 - Match ID 3857277
Procesado 9/64 - Match ID 3857273
Procesado 10/64 - Match ID 3857275
Procesado 11/64 - Match ID 3857261
Procesado 12/64 - Match ID 3857290
Procesado 13/64 - Match ID 3857298
Procesado 14/64 - Match ID 3857285
Procesado 15/64 - Match ID 3857282
Procesado 16/64 - Match ID 3857297
Procesado 17/64 - Match ID 3857294
Procesado 18/64 - Match ID 3857266
Procesado 19/64 - Match ID 3857284
Procesado 20/64 - Match ID 3857254
Procesado 21/64 - Match ID 3857265
Procesado 22/64 - Match ID 3857301
Procesado 23/64 - Match ID 3857268
Procesado 24/64 - Match ID 3857269
Procesado 25/64 - Match ID 3869321
Procesado 26/64 - Match ID 3857292
Procesado 27/64 - Match ID 3857295
Procesado 28/64 - Match ID 3869685
Procesado 29/64 - Match ID 38

,match_id,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,goles_eventos,corners,saques_banda,...,carries,presiones,faltas_cometidas,tarjetas_amarillas,tarjetas_rojas,fecha,local,visita,goles_local,goles_visita
0,3857276,Canada,562,480,85.4,5,0,0,6,23,...,454,135,16,4,0,2022-12-01,Canada,Morocco,1,2
1,3857276,Morocco,393,310,78.9,7,2,2,2,20,...,328,154,15,0,0,2022-12-01,Canada,Morocco,1,2
2,3857271,England,846,746,88.2,13,7,6,8,18,...,660,101,9,0,0,2022-11-21,England,Iran,6,2
3,3857271,Iran,247,163,66.0,8,2,2,0,17,...,190,141,21,2,0,2022-11-21,England,Iran,6,2
4,3857296,Croatia,609,514,84.4,11,4,0,2,27,...,464,164,11,0,0,2022-12-01,Croatia,Belgium,0,0


In [ ]:
# ============================
# PASO 21: Cargar estadísticas de varias competiciones
# ============================

competiciones_a_usar = [
    {"competition_id": 43, "season_id": 106, "nombre": "FIFA World Cup 2022"},
    {"competition_id": 43, "season_id": 3, "nombre": "FIFA World Cup 2018"},
    {"competition_id": 55, "season_id": 43, "nombre": "UEFA Euro 2020"},
    {"competition_id": 55, "season_id": 282, "nombre": "UEFA Euro 2024"},
    {"competition_id": 223, "season_id": 282, "nombre": "Copa America 2024"}
]

def cargar_matches_competicion(competition_id, season_id):
    url_matches = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/{competition_id}/{season_id}.json"
    r = requests.get(url_matches)

    if r.status_code != 200:
        print("No se pudo cargar:", competition_id, season_id)
        return pd.DataFrame()

    matches = r.json()
    return pd.json_normalize(matches)

todas_stats_competiciones = []

for comp in competiciones_a_usar:
    print("\nCargando:", comp["nombre"])

    df_m = cargar_matches_competicion(comp["competition_id"], comp["season_id"])

    if len(df_m) == 0:
        continue

    for i, row in df_m.iterrows():
        match_id = row["match_id"]

        try:
            stats = estadisticas_partido_statsbomb(match_id)

            stats["fecha"] = row["match_date"]
            stats["local"] = row["home_team.home_team_name"]
            stats["visita"] = row["away_team.away_team_name"]
            stats["goles_local"] = row["home_score"]
            stats["goles_visita"] = row["away_score"]
            stats["competicion"] = comp["nombre"]

            todas_stats_competiciones.append(stats)

            print(f"Procesado {i+1}/{len(df_m)} - {comp['nombre']}")

        except Exception as e:
            print("Error en match_id:", match_id, e)

df_stats_eventos = pd.concat(todas_stats_competiciones, ignore_index=True)

print("\nBase final de eventos creada:")
print(df_stats_eventos.shape)

df_stats_eventos.head()


Cargando: FIFA World Cup 2022
Procesado 1/64 - FIFA World Cup 2022
Procesado 2/64 - FIFA World Cup 2022
Procesado 3/64 - FIFA World Cup 2022
Procesado 4/64 - FIFA World Cup 2022
Procesado 5/64 - FIFA World Cup 2022
Procesado 6/64 - FIFA World Cup 2022
Procesado 7/64 - FIFA World Cup 2022
Procesado 8/64 - FIFA World Cup 2022
Procesado 9/64 - FIFA World Cup 2022
Procesado 10/64 - FIFA World Cup 2022
Procesado 11/64 - FIFA World Cup 2022
Procesado 12/64 - FIFA World Cup 2022
Procesado 13/64 - FIFA World Cup 2022
Procesado 14/64 - FIFA World Cup 2022
Procesado 15/64 - FIFA World Cup 2022
Procesado 16/64 - FIFA World Cup 2022
Procesado 17/64 - FIFA World Cup 2022
Procesado 18/64 - FIFA World Cup 2022
Procesado 19/64 - FIFA World Cup 2022
Procesado 20/64 - FIFA World Cup 2022
Procesado 21/64 - FIFA World Cup 2022
Procesado 22/64 - FIFA World Cup 2022
Procesado 23/64 - FIFA World Cup 2022
Procesado 24/64 - FIFA World Cup 2022
Procesado 25/64 - FIFA World Cup 2022
Procesado 26/64 - FIFA World

,match_id,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,goles_eventos,corners,saques_banda,...,presiones,faltas_cometidas,tarjetas_amarillas,tarjetas_rojas,fecha,local,visita,goles_local,goles_visita,competicion
0,3857276,Canada,562,480,85.4,5,0,0,6,23,...,135,16,4,0,2022-12-01,Canada,Morocco,1,2,FIFA World Cup 2022
1,3857276,Morocco,393,310,78.9,7,2,2,2,20,...,154,15,0,0,2022-12-01,Canada,Morocco,1,2,FIFA World Cup 2022
2,3857271,England,846,746,88.2,13,7,6,8,18,...,101,9,0,0,2022-11-21,England,Iran,6,2,FIFA World Cup 2022
3,3857271,Iran,247,163,66.0,8,2,2,0,17,...,141,21,2,0,2022-11-21,England,Iran,6,2,FIFA World Cup 2022
4,3857296,Croatia,609,514,84.4,11,4,0,2,27,...,164,11,0,0,2022-12-01,Croatia,Belgium,0,0,FIFA World Cup 2022


In [ ]:
# ============================
# PASO 22: Revisar estadísticas disponibles
# ============================

print("Cantidad de filas:", len(df_stats_eventos))
print("Competiciones incluidas:")
print(df_stats_eventos["competicion"].value_counts())

print("\nPromedios generales por equipo/partido:")
df_stats_eventos[[
    "pases",
    "pases_completados",
    "precision_pases_%",
    "tiros",
    "tiros_al_arco",
    "corners",
    "saques_banda",
    "faltas_cometidas",
    "tarjetas_amarillas",
    "tarjetas_rojas"
]].mean().round(2)

Cantidad de filas: 524
Competiciones incluidas:
competicion
FIFA World Cup 2022    128
FIFA World Cup 2018    128
UEFA Euro 2020         102
UEFA Euro 2024         102
Copa America 2024       64
Name: count, dtype: int64

Promedios generales por equipo/partido:


,0
pases,511.35
pases_completados,419.47
precision_pases_%,80.46
tiros,12.63
tiros_al_arco,4.23
corners,4.56
saques_banda,18.54
faltas_cometidas,14.00
tarjetas_amarillas,1.79
tarjetas_rojas,0.03


In [ ]:
# ============================
# PASO 23: Crear base ML para estadísticas del partido
# ============================

df_stats_eventos["fecha"] = pd.to_datetime(df_stats_eventos["fecha"])
df_stats_eventos = df_stats_eventos.sort_values("fecha").reset_index(drop=True)

stats_objetivo = [
    "pases",
    "pases_completados",
    "precision_pases_%",
    "tiros",
    "tiros_al_arco",
    "corners",
    "saques_banda",
    "faltas_cometidas",
    "tarjetas_amarillas"
]

historial_stats = defaultdict(list)
filas_stats_ml = []

def promedio_stats_previas(lista_previos, n=5):
    if len(lista_previos) == 0:
        return {stat: df_stats_eventos[stat].mean() for stat in stats_objetivo}

    ultimos = lista_previos[-n:]
    return {
        stat: np.mean([p[stat] for p in ultimos])
        for stat in stats_objetivo
    }

for _, row in df_stats_eventos.iterrows():
    equipo = row["equipo"]

    stats_previas = promedio_stats_previas(historial_stats[equipo], n=5)

    fila = {
        "fecha": row["fecha"],
        "equipo": equipo,
        "competicion": row["competicion"],
        "rival": row["visita"] if equipo == row["local"] else row["local"],
        "es_local": 1 if equipo == row["local"] else 0,
        "es_neutral": 1
    }

    # Variables previas del equipo
    for stat in stats_objetivo:
        fila[f"prev_{stat}"] = stats_previas[stat]

    # Objetivos reales
    for stat in stats_objetivo:
        fila[stat] = row[stat]

    filas_stats_ml.append(fila)

    # Actualizar historial después del partido
    historial_stats[equipo].append({
        stat: row[stat]
        for stat in stats_objetivo
    })

df_stats_ml = pd.DataFrame(filas_stats_ml)

print("Base ML estadísticas:", df_stats_ml.shape)
df_stats_ml.head()

Base ML estadísticas: (524, 24)


,fecha,equipo,competicion,rival,es_local,es_neutral,prev_pases,prev_pases_completados,prev_precision_pases_%,prev_tiros,...,prev_tarjetas_amarillas,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas
0,2018-06-14,Russia,FIFA World Cup 2018,Saudi Arabia,1,1,511.351145,419.467557,80.463931,12.631679,...,1.791985,388,272,70.1,14,7,6,23,23,1
1,2018-06-14,Saudi Arabia,FIFA World Cup 2018,Russia,0,1,511.351145,419.467557,80.463931,12.631679,...,1.791985,572,458,80.1,6,0,2,23,12,1
2,2018-06-15,Spain,FIFA World Cup 2018,Portugal,0,1,511.351145,419.467557,80.463931,12.631679,...,1.791985,780,714,91.5,12,5,5,17,11,1
3,2018-06-15,Uruguay,FIFA World Cup 2018,Egypt,0,1,511.351145,419.467557,80.463931,12.631679,...,1.791985,623,513,82.3,16,4,5,12,7,0
4,2018-06-15,Egypt,FIFA World Cup 2018,Uruguay,1,1,511.351145,419.467557,80.463931,12.631679,...,1.791985,469,344,73.3,8,3,0,22,16,1


In [ ]:
# ============================
# PASO 24: Entrenar modelos para estadísticas
# ============================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import copy

features_stats = [
    "es_local",
    "es_neutral"
] + [f"prev_{stat}" for stat in stats_objetivo]

# Separación temporal
train_stats = df_stats_ml[df_stats_ml["fecha"].dt.year <= 2022].copy()
test_stats = df_stats_ml[df_stats_ml["fecha"].dt.year >= 2024].copy()

X_train_stats = train_stats[features_stats]
X_test_stats = test_stats[features_stats]

modelos_stats = {}
resultados_stats = []

for objetivo in stats_objetivo:
    y_train = train_stats[objetivo]
    y_test = test_stats[objetivo]

    modelo = RandomForestRegressor(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=5,
        random_state=42
    )

    modelo.fit(X_train_stats, y_train)
    pred = modelo.predict(X_test_stats)

    mae = mean_absolute_error(y_test, pred)

    modelos_stats[objetivo] = modelo

    resultados_stats.append({
        "estadistica": objetivo,
        "MAE": round(mae, 2),
        "promedio_real": round(y_test.mean(), 2)
    })

tabla_eval_stats = pd.DataFrame(resultados_stats)
tabla_eval_stats

,estadistica,MAE,promedio_real
0,pases,105.43,492.37
1,pases_completados,102.65,406.63
2,precision_pases_%,4.47,81.05
3,tiros,4.61,12.83
4,tiros_al_arco,2.06,4.31
5,corners,2.28,4.69
6,saques_banda,4.62,17.73
7,faltas_cometidas,3.67,13.81
8,tarjetas_amarillas,1.14,2.13


In [ ]:
# ============================
# PASO 25: Predecir estadísticas de un equipo
# ============================

def crear_features_stats_equipo(equipo, es_local=1, es_neutral=1):
    stats_previas = promedio_stats_previas(historial_stats[equipo], n=5)

    fila = {
        "es_local": es_local,
        "es_neutral": es_neutral
    }

    for stat in stats_objetivo:
        fila[f"prev_{stat}"] = stats_previas[stat]

    return pd.DataFrame([fila])[features_stats]


def predecir_stats_equipo(equipo, es_local=1, es_neutral=1):
    X_equipo = crear_features_stats_equipo(equipo, es_local, es_neutral)

    predicciones = {}

    for stat, modelo in modelos_stats.items():
        valor = modelo.predict(X_equipo)[0]
        predicciones[stat] = round(max(0, valor), 1)

    return predicciones


def predecir_stats_partido(equipo_1, equipo_2):
    stats_1 = predecir_stats_equipo(equipo_1, es_local=1, es_neutral=1)
    stats_2 = predecir_stats_equipo(equipo_2, es_local=0, es_neutral=1)

    tabla = pd.DataFrame([
        {"equipo": equipo_1, **stats_1},
        {"equipo": equipo_2, **stats_2}
    ])

    return tabla

In [ ]:
predecir_stats_partido("Mexico", "Ecuador")

,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas
0,Mexico,461.1,366.0,77.6,12.7,5.3,5.2,20.3,15.7,2.1
1,Ecuador,417.5,322.9,77.2,10.3,2.7,3.3,17.9,14.3,1.7


In [ ]:
predecir_stats_partido("England", "DR Congo")

,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas
0,England,577.9,491.3,83.9,15.2,5.8,4.6,18.8,13.3,1.7
1,DR Congo,474.0,379.8,78.7,11.1,3.4,3.8,18.0,14.6,1.5


In [ ]:
# ============================
# PASO 27: Comparar predicción vs estadística real
# ============================

pred_mex_ecu = predecir_stats_partido("Mexico", "Ecuador")

real_mex_ecu = pd.DataFrame([
    {
        "equipo": "Mexico",
        "pases": 306,
        "precision_pases_%": 83,
        "tiros": 15,
        "tiros_al_arco": 3,
        "corners": 3,
        "faltas_cometidas": 10,
        "tarjetas_amarillas": 0
    },
    {
        "equipo": "Ecuador",
        "pases": 416,
        "precision_pases_%": 84,
        "tiros": 8,
        "tiros_al_arco": 1,
        "corners": 8,
        "faltas_cometidas": 14,
        "tarjetas_amarillas": 3
    }
])

comparacion = pred_mex_ecu.merge(real_mex_ecu, on="equipo", suffixes=("_pred", "_real"))

stats_comparar = [
    "pases",
    "precision_pases_%",
    "tiros",
    "tiros_al_arco",
    "corners",
    "faltas_cometidas",
    "tarjetas_amarillas"
]

for stat in stats_comparar:
    comparacion[f"error_{stat}"] = comparacion[f"{stat}_pred"] - comparacion[f"{stat}_real"]
    comparacion[f"error_abs_{stat}"] = abs(comparacion[f"error_{stat}"])

comparacion

,equipo,pases_pred,pases_completados,precision_pases_%_pred,tiros_pred,tiros_al_arco_pred,corners_pred,saques_banda,faltas_cometidas_pred,tarjetas_amarillas_pred,...,error_tiros,error_abs_tiros,error_tiros_al_arco,error_abs_tiros_al_arco,error_corners,error_abs_corners,error_faltas_cometidas,error_abs_faltas_cometidas,error_tarjetas_amarillas,error_abs_tarjetas_amarillas
0,Mexico,461.1,366.0,77.6,12.7,5.3,5.2,20.3,15.7,2.1,...,-2.3,2.3,2.3,2.3,2.2,2.2,5.7,5.7,2.1,2.1
1,Ecuador,417.5,322.9,77.2,10.3,2.7,3.3,17.9,14.3,1.7,...,2.3,2.3,1.7,1.7,-4.7,4.7,0.3,0.3,-1.3,1.3


In [ ]:
# ============================
# PASO 28: Resumen de errores
# ============================

errores_resumen = []

for _, row in comparacion.iterrows():
    for stat in stats_comparar:
        errores_resumen.append({
            "equipo": row["equipo"],
            "estadistica": stat,
            "predicho": round(row[f"{stat}_pred"], 1),
            "real": row[f"{stat}_real"],
            "error": round(row[f"error_{stat}"], 1),
            "error_absoluto": round(row[f"error_abs_{stat}"], 1)
        })

pd.DataFrame(errores_resumen).sort_values("error_absoluto", ascending=False)

,equipo,estadistica,predicho,real,error,error_absoluto
0,Mexico,pases,461.1,306,155.1,155.1
8,Ecuador,precision_pases_%,77.2,84,-6.8,6.8
5,Mexico,faltas_cometidas,15.7,10,5.7,5.7
1,Mexico,precision_pases_%,77.6,83,-5.4,5.4
11,Ecuador,corners,3.3,8,-4.7,4.7
2,Mexico,tiros,12.7,15,-2.3,2.3
9,Ecuador,tiros,10.3,8,2.3,2.3
3,Mexico,tiros_al_arco,5.3,3,2.3,2.3
4,Mexico,corners,5.2,3,2.2,2.2
6,Mexico,tarjetas_amarillas,2.1,0,2.1,2.1


In [ ]:
# ============================
# REPORTE COMPLETO DE UN PARTIDO
# Goles + probabilidades + estadísticas
# ============================

from scipy.stats import poisson
import pandas as pd
import numpy as np

def probabilidades_goles_extra(equipo_1, equipo_2):
    base = predecir_partido(equipo_1, equipo_2, neutral=1).iloc[0]

    g1 = base["goles_esperados_equipo_1"]
    g2 = base["goles_esperados_equipo_2"]
    total_goles = g1 + g2

    datos = {
        "prob_marca_" + equipo_1: round((1 - poisson.pmf(0, g1)) * 100, 1),
        "prob_marca_" + equipo_2: round((1 - poisson.pmf(0, g2)) * 100, 1),
        "prob_ambos_marcan": round(((1 - poisson.pmf(0, g1)) * (1 - poisson.pmf(0, g2))) * 100, 1),
        "prob_menos_1_5_goles": round(poisson.cdf(1, total_goles) * 100, 1),
        "prob_mas_1_5_goles": round((1 - poisson.cdf(1, total_goles)) * 100, 1),
        "prob_menos_2_5_goles": round(poisson.cdf(2, total_goles) * 100, 1),
        "prob_mas_2_5_goles": round((1 - poisson.cdf(2, total_goles)) * 100, 1),
        "prob_menos_3_5_goles": round(poisson.cdf(3, total_goles) * 100, 1),
        "prob_mas_3_5_goles": round((1 - poisson.cdf(3, total_goles)) * 100, 1),
    }

    return pd.DataFrame([datos])


def matriz_marcadores_probables(equipo_1, equipo_2, max_goles=5):
    base = predecir_partido(equipo_1, equipo_2, neutral=1).iloc[0]

    g1 = base["goles_esperados_equipo_1"]
    g2 = base["goles_esperados_equipo_2"]

    marcadores = []

    for goles_1 in range(max_goles + 1):
        for goles_2 in range(max_goles + 1):
            prob = poisson.pmf(goles_1, g1) * poisson.pmf(goles_2, g2)

            marcadores.append({
                "marcador": f"{equipo_1} {goles_1} - {goles_2} {equipo_2}",
                "probabilidad_%": round(prob * 100, 2)
            })

    return pd.DataFrame(marcadores).sort_values("probabilidad_%", ascending=False).head(10)


def agregar_posesion_aprox(tabla_stats):
    tabla = tabla_stats.copy()

    total_pases = tabla["pases"].sum()

    if total_pases > 0:
        tabla["posesion_aprox_%"] = round((tabla["pases"] / total_pases) * 100, 1)
    else:
        tabla["posesion_aprox_%"] = 50

    return tabla


def reporte_partido_completo(equipo_1, equipo_2):
    print("====================================")
    print(f"REPORTE DEL PARTIDO: {equipo_1} vs {equipo_2}")
    print("====================================")

    print("\n1) Predicción de goles y resultado")
    pred_goles = predecir_partido(equipo_1, equipo_2, neutral=1)
    display(pred_goles)

    print("\n2) Probabilidades de goles")
    probs_extra = probabilidades_goles_extra(equipo_1, equipo_2)
    display(probs_extra)

    print("\n3) Estadísticas estimadas del partido")
    stats = predecir_stats_partido(equipo_1, equipo_2)
    stats = agregar_posesion_aprox(stats)
    display(stats)

    print("\n4) Marcadores más probables")
    marcadores = matriz_marcadores_probables(equipo_1, equipo_2, max_goles=5)
    display(marcadores)

    return pred_goles, probs_extra, stats, marcadores

In [ ]:
reporte_partido_completo("England", "DR Congo")

REPORTE DEL PARTIDO: England vs DR Congo

1) Predicción de goles y resultado


,equipo_1,equipo_2,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,marcador_mas_probable,prob_marcador
0,England,DR Congo,1.92,0.62,68.3,20.4,11.2,1 - 0,15.1



2) Probabilidades de goles


,prob_marca_England,prob_marca_DR Congo,prob_ambos_marcan,prob_menos_1_5_goles,prob_mas_1_5_goles,prob_menos_2_5_goles,prob_mas_2_5_goles,prob_menos_3_5_goles,prob_mas_3_5_goles
0,85.3,46.2,39.4,27.9,72.1,53.4,46.6,74.9,25.1



3) Estadísticas estimadas del partido


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,England,577.9,491.3,83.9,15.2,5.8,4.6,18.8,13.3,1.7,54.9
1,DR Congo,474.0,379.8,78.7,11.1,3.4,3.8,18.0,14.6,1.5,45.1



4) Marcadores más probables


,marcador,probabilidad_%
6,England 1 - 0 DR Congo,15.14
12,England 2 - 0 DR Congo,14.54
7,England 1 - 1 DR Congo,9.39
18,England 3 - 0 DR Congo,9.30
13,England 2 - 1 DR Congo,9.01
0,England 0 - 0 DR Congo,7.89
19,England 3 - 1 DR Congo,5.77
1,England 0 - 1 DR Congo,4.89
24,England 4 - 0 DR Congo,4.47
8,England 1 - 2 DR Congo,2.91


(  equipo_1  equipo_2  goles_esperados_equipo_1  goles_esperados_equipo_2  \
 0  England  DR Congo                      1.92                      0.62   
 
    prob_gana_equipo_1  prob_empate  prob_gana_equipo_2 marcador_mas_probable  \
 0                68.3         20.4                11.2                 1 - 0   
 
    prob_marcador  
 0           15.1  ,
    prob_marca_England  prob_marca_DR Congo  prob_ambos_marcan  \
 0                85.3                 46.2               39.4   
 
    prob_menos_1_5_goles  prob_mas_1_5_goles  prob_menos_2_5_goles  \
 0                  27.9                72.1                  53.4   
 
    prob_mas_2_5_goles  prob_menos_3_5_goles  prob_mas_3_5_goles  
 0                46.6                  74.9                25.1  ,
      equipo  pases  pases_completados  precision_pases_%  tiros  \
 0   England  577.9              491.3               83.9   15.2   
 1  DR Congo  474.0              379.8               78.7   11.1   
 
    tiros_al_arco  cor

In [ ]:
reporte_partido_completo("Mexico", "Ecuador")

REPORTE DEL PARTIDO: Mexico vs Ecuador

1) Predicción de goles y resultado


,equipo_1,equipo_2,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,marcador_mas_probable,prob_marcador
0,Mexico,Ecuador,1.57,0.67,59.1,25.2,15.7,1 - 0,16.7



2) Probabilidades de goles


,prob_marca_Mexico,prob_marca_Ecuador,prob_ambos_marcan,prob_menos_1_5_goles,prob_mas_1_5_goles,prob_menos_2_5_goles,prob_mas_2_5_goles,prob_menos_3_5_goles,prob_mas_3_5_goles
0,79.2,48.8,38.7,34.5,65.5,61.2,38.8,81.1,18.9



3) Estadísticas estimadas del partido


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,Mexico,461.1,366.0,77.6,12.7,5.3,5.2,20.3,15.7,2.1,52.5
1,Ecuador,417.5,322.9,77.2,10.3,2.7,3.3,17.9,14.3,1.7,47.5



4) Marcadores más probables


,marcador,probabilidad_%
6,Mexico 1 - 0 Ecuador,16.71
12,Mexico 2 - 0 Ecuador,13.12
7,Mexico 1 - 1 Ecuador,11.20
0,Mexico 0 - 0 Ecuador,10.65
13,Mexico 2 - 1 Ecuador,8.79
1,Mexico 0 - 1 Ecuador,7.13
18,Mexico 3 - 0 Ecuador,6.87
19,Mexico 3 - 1 Ecuador,4.60
8,Mexico 1 - 2 Ecuador,3.75
14,Mexico 2 - 2 Ecuador,2.94


(  equipo_1 equipo_2  goles_esperados_equipo_1  goles_esperados_equipo_2  \
 0   Mexico  Ecuador                      1.57                      0.67   
 
    prob_gana_equipo_1  prob_empate  prob_gana_equipo_2 marcador_mas_probable  \
 0                59.1         25.2                15.7                 1 - 0   
 
    prob_marcador  
 0           16.7  ,
    prob_marca_Mexico  prob_marca_Ecuador  prob_ambos_marcan  \
 0               79.2                48.8               38.7   
 
    prob_menos_1_5_goles  prob_mas_1_5_goles  prob_menos_2_5_goles  \
 0                  34.5                65.5                  61.2   
 
    prob_mas_2_5_goles  prob_menos_3_5_goles  prob_mas_3_5_goles  
 0                38.8                  81.1                18.9  ,
     equipo  pases  pases_completados  precision_pases_%  tiros  tiros_al_arco  \
 0   Mexico  461.1              366.0               77.6   12.7            5.3   
 1  Ecuador  417.5              322.9               77.2   10.3       

In [ ]:
def resumen_simple_partido(equipo_1, equipo_2):
    pred = predecir_partido(equipo_1, equipo_2, neutral=1).iloc[0]
    probs = probabilidades_goles_extra(equipo_1, equipo_2).iloc[0]
    stats = agregar_posesion_aprox(predecir_stats_partido(equipo_1, equipo_2))

    resumen = pd.DataFrame([
        ["Favorito", equipo_1 if pred["prob_gana_equipo_1"] > pred["prob_gana_equipo_2"] else equipo_2],
        ["Probabilidad gana favorito", max(pred["prob_gana_equipo_1"], pred["prob_gana_equipo_2"])],
        ["Marcador más probable", pred["marcador_mas_probable"]],
        [f"Probabilidad marca {equipo_1}", probs[f"prob_marca_{equipo_1}"]],
        [f"Probabilidad marca {equipo_2}", probs[f"prob_marca_{equipo_2}"]],
        ["Ambos marcan", probs["prob_ambos_marcan"]],
        ["Más de 1.5 goles", probs["prob_mas_1_5_goles"]],
        ["Más de 2.5 goles", probs["prob_mas_2_5_goles"]],
        ["Menos de 3.5 goles", probs["prob_menos_3_5_goles"]],
        ["Corners totales aprox.", round(stats["corners"].sum(), 1)],
        ["Tiros totales aprox.", round(stats["tiros"].sum(), 1)],
        ["Tarjetas amarillas aprox.", round(stats["tarjetas_amarillas"].sum(), 1)]
    ], columns=["Indicador", "Valor"])

    return resumen

resumen_simple_partido("England", "DR Congo")

,Indicador,Valor
0,Favorito,England
1,Probabilidad gana favorito,68.3
2,Marcador más probable,1 - 0
3,Probabilidad marca England,85.3
4,Probabilidad marca DR Congo,46.2
5,Ambos marcan,39.4
6,Más de 1.5 goles,72.1
7,Más de 2.5 goles,46.6
8,Menos de 3.5 goles,74.9
9,Corners totales aprox.,8.4


In [ ]:
# ============================
# SIMULADOR MONTE CARLO DE GOLES
# ============================

import numpy as np
import pandas as pd

def simular_partido_montecarlo(equipo_1, equipo_2, n_simulaciones=100000):
    pred = predecir_partido(equipo_1, equipo_2, neutral=1).iloc[0]

    g1 = pred["goles_esperados_equipo_1"]
    g2 = pred["goles_esperados_equipo_2"]

    # Simular goles usando Poisson
    goles_1 = np.random.poisson(g1, n_simulaciones)
    goles_2 = np.random.poisson(g2, n_simulaciones)

    gana_1 = np.mean(goles_1 > goles_2) * 100
    empate = np.mean(goles_1 == goles_2) * 100
    gana_2 = np.mean(goles_2 > goles_1) * 100

    ambos_marcan = np.mean((goles_1 > 0) & (goles_2 > 0)) * 100
    mas_15 = np.mean((goles_1 + goles_2) > 1.5) * 100
    mas_25 = np.mean((goles_1 + goles_2) > 2.5) * 100
    menos_35 = np.mean((goles_1 + goles_2) < 3.5) * 100

    # Marcadores más frecuentes
    marcadores = pd.DataFrame({
        "marcador": [f"{a}-{b}" for a, b in zip(goles_1, goles_2)]
    })

    top_marcadores = (
        marcadores["marcador"]
        .value_counts(normalize=True)
        .head(10)
        .reset_index()
    )

    top_marcadores.columns = ["marcador", "probabilidad_%"]
    top_marcadores["probabilidad_%"] = (top_marcadores["probabilidad_%"] * 100).round(2)

    resumen = pd.DataFrame([{
        "equipo_1": equipo_1,
        "equipo_2": equipo_2,
        "simulaciones": n_simulaciones,
        "goles_esperados_equipo_1": round(g1, 2),
        "goles_esperados_equipo_2": round(g2, 2),
        "prob_gana_equipo_1": round(gana_1, 1),
        "prob_empate": round(empate, 1),
        "prob_gana_equipo_2": round(gana_2, 1),
        "prob_ambos_marcan": round(ambos_marcan, 1),
        "prob_mas_1_5_goles": round(mas_15, 1),
        "prob_mas_2_5_goles": round(mas_25, 1),
        "prob_menos_3_5_goles": round(menos_35, 1)
    }])

    return resumen, top_marcadores

In [ ]:
resumen_bel_sen, marcadores_bel_sen = simular_partido_montecarlo(
    "Belgium",
    "Senegal",
    n_simulaciones=100000
)

display(resumen_bel_sen)
display(marcadores_bel_sen)

,equipo_1,equipo_2,simulaciones,goles_esperados_equipo_1,goles_esperados_equipo_2,prob_gana_equipo_1,prob_empate,prob_gana_equipo_2,prob_ambos_marcan,prob_mas_1_5_goles,prob_mas_2_5_goles,prob_menos_3_5_goles
0,Belgium,Senegal,100000,1.63,0.79,57.4,24.7,17.9,43.8,69.6,43.6,77.3


,marcador,probabilidad_%
0,1-0,14.47
1,2-0,11.97
2,1-1,11.31
3,2-1,9.18
4,0-0,8.98
5,0-1,6.99
6,3-0,6.43
7,3-1,5.10
8,1-2,4.57
9,2-2,3.82


In [ ]:
# ============================
# REPORTE TOTAL: GOLES + STATS
# ============================

from scipy.stats import poisson
import pandas as pd
import numpy as np

def reporte_total_partido(equipo_1, equipo_2):

    # Validar funciones necesarias
    necesarias = ["predecir_partido", "predecir_stats_partido"]
    for f in necesarias:
        if f not in globals():
            raise NameError(f"Falta ejecutar la función: {f}. Debes volver a ejecutar el notebook desde el inicio o desde el Paso 9.")

    # 1) Predicción de goles
    pred = predecir_partido(equipo_1, equipo_2, neutral=1).iloc[0]

    g1 = pred["goles_esperados_equipo_1"]
    g2 = pred["goles_esperados_equipo_2"]
    total_goles = g1 + g2

    resumen_goles = pd.DataFrame([
        ["Goles esperados " + equipo_1, round(g1, 2)],
        ["Goles esperados " + equipo_2, round(g2, 2)],
        ["Prob. gana " + equipo_1, str(pred["prob_gana_equipo_1"]) + "%"],
        ["Prob. empate", str(pred["prob_empate"]) + "%"],
        ["Prob. gana " + equipo_2, str(pred["prob_gana_equipo_2"]) + "%"],
        ["Marcador más probable", pred["marcador_mas_probable"]],
        ["Prob. marcador exacto", str(pred["prob_marcador"]) + "%"],
        ["Prob. " + equipo_1 + " marca", str(round((1 - poisson.pmf(0, g1)) * 100, 1)) + "%"],
        ["Prob. " + equipo_2 + " marca", str(round((1 - poisson.pmf(0, g2)) * 100, 1)) + "%"],
        ["Ambos marcan", str(round(((1 - poisson.pmf(0, g1)) * (1 - poisson.pmf(0, g2))) * 100, 1)) + "%"],
        ["Más de 1.5 goles", str(round((1 - poisson.cdf(1, total_goles)) * 100, 1)) + "%"],
        ["Más de 2.5 goles", str(round((1 - poisson.cdf(2, total_goles)) * 100, 1)) + "%"],
        ["Menos de 3.5 goles", str(round(poisson.cdf(3, total_goles) * 100, 1)) + "%"],
    ], columns=["Indicador", "Valor"])

    # 2) Estadísticas estimadas
    stats = predecir_stats_partido(equipo_1, equipo_2)

    # Posesión aproximada por pases
    total_pases = stats["pases"].sum()
    stats["posesion_aprox_%"] = round((stats["pases"] / total_pases) * 100, 1)

    # 3) Totales del partido
    totales = pd.DataFrame([
        ["Pases totales", round(stats["pases"].sum(), 1)],
        ["Pases completados totales", round(stats["pases_completados"].sum(), 1)],
        ["Tiros totales", round(stats["tiros"].sum(), 1)],
        ["Tiros al arco totales", round(stats["tiros_al_arco"].sum(), 1)],
        ["Corners totales", round(stats["corners"].sum(), 1)],
        ["Saques de banda totales", round(stats["saques_banda"].sum(), 1)],
        ["Faltas totales", round(stats["faltas_cometidas"].sum(), 1)],
        ["Amarillas totales", round(stats["tarjetas_amarillas"].sum(), 1)],
    ], columns=["Estadística total", "Estimación"])

    # 4) Rangos simples usando margen aproximado
    rangos = []

    margen_aprox = {
        "pases": 90,
        "tiros": 4,
        "tiros_al_arco": 2,
        "corners": 2,
        "saques_banda": 5,
        "faltas_cometidas": 4,
        "tarjetas_amarillas": 1.5
    }

    for _, row in stats.iterrows():
        equipo = row["equipo"]
        for stat, margen in margen_aprox.items():
            valor = row[stat]
            rangos.append({
                "equipo": equipo,
                "estadística": stat,
                "estimado": round(valor, 1),
                "rango_aprox": f"{round(max(0, valor - margen), 1)} - {round(valor + margen, 1)}"
            })

    rangos = pd.DataFrame(rangos)

    # 5) Marcadores más probables
    marcadores = []
    for a in range(6):
        for b in range(6):
            prob = poisson.pmf(a, g1) * poisson.pmf(b, g2)
            marcadores.append({
                "marcador": f"{equipo_1} {a} - {b} {equipo_2}",
                "probabilidad_%": round(prob * 100, 2)
            })

    marcadores = pd.DataFrame(marcadores).sort_values("probabilidad_%", ascending=False).head(10)

    print("====================================")
    print(f"REPORTE TOTAL: {equipo_1} vs {equipo_2}")
    print("====================================")

    print("\n1) GOLES Y RESULTADO")
    display(resumen_goles)

    print("\n2) ESTADÍSTICAS POR EQUIPO")
    display(stats)

    print("\n3) TOTALES DEL PARTIDO")
    display(totales)

    print("\n4) RANGOS APROXIMADOS")
    display(rangos)

    print("\n5) MARCADORES MÁS PROBABLES")
    display(marcadores)

    return resumen_goles, stats, totales, rangos, marcadores

In [ ]:
# ============================
# CAPA DE AJUSTE DEL MODELO
# ============================

import pandas as pd
import numpy as np

factores_ajuste = {
    "Pases totales": 0.80,
    "Pases completados totales": 0.84,
    "Tiros totales": 0.82,
    "Tiros al arco totales": 0.82,
    "Corners totales": 0.95,
    "Saques de banda totales": 1.00,
    "Faltas totales": 0.73,
    "Amarillas totales": 0.85
}

margenes_conservadores = {
    "Pases totales": 120,
    "Pases completados totales": 110,
    "Tiros totales": 5,
    "Tiros al arco totales": 3,
    "Corners totales": 3,
    "Saques de banda totales": 6,
    "Faltas totales": 5,
    "Amarillas totales": 1.5
}

confianza_variable = {
    "Pases totales": "Baja",
    "Pases completados totales": "Baja",
    "Tiros totales": "Media-alta",
    "Tiros al arco totales": "Media",
    "Corners totales": "Media-alta",
    "Saques de banda totales": "Media",
    "Faltas totales": "Media",
    "Amarillas totales": "Media-alta"
}

def ajustar_totales_modelo(totales):
    df = totales.copy()

    df["factor_ajuste"] = df["Estadística total"].map(factores_ajuste).fillna(1.00)
    df["estimacion_ajustada"] = (df["Estimación"] * df["factor_ajuste"]).round(1)

    df["margen"] = df["Estadística total"].map(margenes_conservadores).fillna(0)

    df["rango_ajustado"] = df.apply(
        lambda row: f"{round(max(0, row['estimacion_ajustada'] - row['margen']), 1)} - {round(row['estimacion_ajustada'] + row['margen'], 1)}",
        axis=1
    )

    df["confianza"] = df["Estadística total"].map(confianza_variable).fillna("Media")

    return df

In [ ]:
def evaluar_senal_total(totales_ajustados, estadistica, linea, tipo="over"):
    fila = totales_ajustados[totales_ajustados["Estadística total"] == estadistica]

    if len(fila) == 0:
        return "No encontré esa estadística. Revisa el nombre exacto."

    estimacion = fila.iloc[0]["estimacion_ajustada"]
    margen = fila.iloc[0]["margen"]
    confianza_base = fila.iloc[0]["confianza"]

    if tipo == "over":
        diferencia = estimacion - linea
    elif tipo == "under":
        diferencia = linea - estimacion
    else:
        return "tipo debe ser 'over' o 'under'"

    if diferencia >= margen * 0.8:
        fuerza = "Fuerte"
    elif diferencia >= margen * 0.4:
        fuerza = "Media"
    elif diferencia > 0:
        fuerza = "Leve"
    else:
        fuerza = "No respaldada"

    resultado = pd.DataFrame([{
        "estadistica": estadistica,
        "tipo": tipo,
        "linea": linea,
        "estimacion_ajustada": estimacion,
        "diferencia_a_favor": round(diferencia, 2),
        "rango_ajustado": fila.iloc[0]["rango_ajustado"],
        "confianza_variable": confianza_base,
        "fuerza_senal": fuerza
    }])

    return resultado

In [ ]:
goles_esp_bel, stats_esp_bel, totales_esp_bel, rangos_esp_bel, marcadores_esp_bel = reporte_total_partido("Spain", "Belgium")

totales_esp_bel_ajustados = ajustar_totales_modelo(totales_esp_bel)
display(totales_esp_bel_ajustados)

REPORTE TOTAL: Spain vs Belgium

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados Spain,1.96
1,Goles esperados Belgium,0.63
2,Prob. gana Spain,68.8%
3,Prob. empate,20.0%
4,Prob. gana Belgium,11.1%
5,Marcador más probable,1 - 0
6,Prob. marcador exacto,14.6%
7,Prob. Spain marca,85.9%
8,Prob. Belgium marca,46.7%
9,Ambos marcan,40.2%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,Spain,594.9,502.0,84.1,18.3,7.7,6.9,20.7,10.6,1.6,48.9
1,Belgium,621.5,527.7,84.8,14.9,4.6,5.9,19.7,13.5,1.2,51.1



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,Spain,pases,594.9,504.9 - 684.9
1,Spain,tiros,18.3,14.3 - 22.3
2,Spain,tiros_al_arco,7.7,5.7 - 9.7
3,Spain,corners,6.9,4.9 - 8.9
4,Spain,saques_banda,20.7,15.7 - 25.7
5,Spain,faltas_cometidas,10.6,6.6 - 14.6
6,Spain,tarjetas_amarillas,1.6,0.1 - 3.1
7,Belgium,pases,621.5,531.5 - 711.5
8,Belgium,tiros,14.9,10.9 - 18.9
9,Belgium,tiros_al_arco,4.6,2.6 - 6.6



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
6,Spain 1 - 0 Belgium,14.70
12,Spain 2 - 0 Belgium,14.41
18,Spain 3 - 0 Belgium,9.41
7,Spain 1 - 1 Belgium,9.26
13,Spain 2 - 1 Belgium,9.08
0,Spain 0 - 0 Belgium,7.50
19,Spain 3 - 1 Belgium,5.93
1,Spain 0 - 1 Belgium,4.73
24,Spain 4 - 0 Belgium,4.61
8,Spain 1 - 2 Belgium,2.92


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1216.4,0.80,973.1,120.0,853.1 - 1093.1,Baja
1,Pases completados totales,1029.7,0.84,864.9,110.0,754.9 - 974.9,Baja
2,Tiros totales,33.2,0.82,27.2,5.0,22.2 - 32.2,Media-alta
3,Tiros al arco totales,12.3,0.82,10.1,3.0,7.1 - 13.1,Media
4,Corners totales,12.8,0.95,12.2,3.0,9.2 - 15.2,Media-alta
5,Saques de banda totales,40.4,1.00,40.4,6.0,34.4 - 46.4,Media
6,Faltas totales,24.1,0.73,17.6,5.0,12.6 - 22.6,Media
7,Amarillas totales,2.8,0.85,2.4,1.5,0.9 - 3.9,Media-alta


In [ ]:
goles_eng_nor, stats_eng_nor, totales_eng_nor, rangos_eng_nor, marcadores_eng_nor = reporte_total_partido("England", "Norway")

totales_eng_nor_ajustados = ajustar_totales_modelo(totales_eng_nor)
display(totales_eng_nor_ajustados)

REPORTE TOTAL: England vs Norway

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados England,1.62
1,Goles esperados Norway,0.98
2,Prob. gana England,52.2%
3,Prob. empate,24.8%
4,Prob. gana Norway,22.9%
5,Marcador más probable,1 - 0
6,Prob. marcador exacto,12.0%
7,Prob. England marca,80.2%
8,Prob. Norway marca,62.5%
9,Ambos marcan,50.1%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,England,577.9,491.3,83.9,15.2,5.8,4.6,18.8,13.3,1.7,54.9
1,Norway,474.0,379.8,78.7,11.1,3.4,3.8,18.0,14.6,1.5,45.1



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,England,pases,577.9,487.9 - 667.9
1,England,tiros,15.2,11.2 - 19.2
2,England,tiros_al_arco,5.8,3.8 - 7.8
3,England,corners,4.6,2.6 - 6.6
4,England,saques_banda,18.8,13.8 - 23.8
5,England,faltas_cometidas,13.3,9.3 - 17.3
6,England,tarjetas_amarillas,1.7,0.2 - 3.2
7,Norway,pases,474.0,384.0 - 564.0
8,Norway,tiros,11.1,7.1 - 15.1
9,Norway,tiros_al_arco,3.4,1.4 - 5.4



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
6,England 1 - 0 Norway,12.03
7,England 1 - 1 Norway,11.79
12,England 2 - 0 Norway,9.75
13,England 2 - 1 Norway,9.55
0,England 0 - 0 Norway,7.43
1,England 0 - 1 Norway,7.28
8,England 1 - 2 Norway,5.78
18,England 3 - 0 Norway,5.26
19,England 3 - 1 Norway,5.16
14,England 2 - 2 Norway,4.68


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1051.9,0.80,841.5,120.0,721.5 - 961.5,Baja
1,Pases completados totales,871.1,0.84,731.7,110.0,621.7 - 841.7,Baja
2,Tiros totales,26.3,0.82,21.6,5.0,16.6 - 26.6,Media-alta
3,Tiros al arco totales,9.2,0.82,7.5,3.0,4.5 - 10.5,Media
4,Corners totales,8.4,0.95,8.0,3.0,5.0 - 11.0,Media-alta
5,Saques de banda totales,36.8,1.00,36.8,6.0,30.8 - 42.8,Media
6,Faltas totales,27.9,0.73,20.4,5.0,15.4 - 25.4,Media
7,Amarillas totales,3.2,0.85,2.7,1.5,1.2 - 4.2,Media-alta


In [ ]:
goles_nor_eng, stats_nor_eng, totales_nor_eng, rangos_nor_eng, marcadores_nor_eng = reporte_total_partido("Norway", "England")

totales_nor_eng_ajustados = ajustar_totales_modelo(totales_nor_eng)
display(totales_nor_eng_ajustados)

REPORTE TOTAL: Norway vs England

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados Norway,0.98
1,Goles esperados England,1.58
2,Prob. gana Norway,23.4%
3,Prob. empate,25.2%
4,Prob. gana England,51.3%
5,Marcador más probable,0 - 1
6,Prob. marcador exacto,12.3%
7,Prob. Norway marca,62.5%
8,Prob. England marca,79.4%
9,Ambos marcan,49.6%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,Norway,484.3,390.5,79.1,11.5,3.6,4.1,18.0,14.6,1.5,45.6
1,England,577.7,491.0,83.9,15.1,5.8,4.7,18.8,13.5,1.8,54.4



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,Norway,pases,484.3,394.3 - 574.3
1,Norway,tiros,11.5,7.5 - 15.5
2,Norway,tiros_al_arco,3.6,1.6 - 5.6
3,Norway,corners,4.1,2.1 - 6.1
4,Norway,saques_banda,18.0,13.0 - 23.0
5,Norway,faltas_cometidas,14.6,10.6 - 18.6
6,Norway,tarjetas_amarillas,1.5,0 - 3.0
7,England,pases,577.7,487.7 - 667.7
8,England,tiros,15.1,11.1 - 19.1
9,England,tiros_al_arco,5.8,3.8 - 7.8



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
1,Norway 0 - 1 England,12.21
7,Norway 1 - 1 England,11.97
2,Norway 0 - 2 England,9.65
8,Norway 1 - 2 England,9.46
0,Norway 0 - 0 England,7.73
6,Norway 1 - 0 England,7.58
13,Norway 2 - 1 England,5.87
3,Norway 0 - 3 England,5.08
9,Norway 1 - 3 England,4.98
14,Norway 2 - 2 England,4.63


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1062.0,0.80,849.6,120.0,729.6 - 969.6,Baja
1,Pases completados totales,881.5,0.84,740.5,110.0,630.5 - 850.5,Baja
2,Tiros totales,26.6,0.82,21.8,5.0,16.8 - 26.8,Media-alta
3,Tiros al arco totales,9.4,0.82,7.7,3.0,4.7 - 10.7,Media
4,Corners totales,8.8,0.95,8.4,3.0,5.4 - 11.4,Media-alta
5,Saques de banda totales,36.8,1.00,36.8,6.0,30.8 - 42.8,Media
6,Faltas totales,28.1,0.73,20.5,5.0,15.5 - 25.5,Media
7,Amarillas totales,3.3,0.85,2.8,1.5,1.3 - 4.3,Media-alta


In [ ]:
goles_arg_sui, stats_arg_sui, totales_arg_sui, rangos_arg_sui, marcadores_arg_sui = reporte_total_partido("Argentina", "Switzerland")

totales_arg_sui_ajustados = ajustar_totales_modelo(totales_arg_sui)
display(totales_arg_sui_ajustados)

REPORTE TOTAL: Argentina vs Switzerland

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados Argentina,1.89
1,Goles esperados Switzerland,0.66
2,Prob. gana Argentina,66.8%
3,Prob. empate,21.0%
4,Prob. gana Switzerland,12.1%
5,Marcador más probable,1 - 0
6,Prob. marcador exacto,14.8%
7,Prob. Argentina marca,84.9%
8,Prob. Switzerland marca,48.3%
9,Ambos marcan,41.0%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,Argentina,575.1,494.9,85.7,12.8,4.3,5.1,18.2,12.9,1.0,50.8
1,Switzerland,556.4,458.0,80.8,12.5,4.1,4.3,19.6,13.5,1.4,49.2



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,Argentina,pases,575.1,485.1 - 665.1
1,Argentina,tiros,12.8,8.8 - 16.8
2,Argentina,tiros_al_arco,4.3,2.3 - 6.3
3,Argentina,corners,5.1,3.1 - 7.1
4,Argentina,saques_banda,18.2,13.2 - 23.2
5,Argentina,faltas_cometidas,12.9,8.9 - 16.9
6,Argentina,tarjetas_amarillas,1.0,0 - 2.5
7,Switzerland,pases,556.4,466.4 - 646.4
8,Switzerland,tiros,12.5,8.5 - 16.5
9,Switzerland,tiros_al_arco,4.1,2.1 - 6.1



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
6,Argentina 1 - 0 Switzerland,14.76
12,Argentina 2 - 0 Switzerland,13.95
7,Argentina 1 - 1 Switzerland,9.74
13,Argentina 2 - 1 Switzerland,9.20
18,Argentina 3 - 0 Switzerland,8.79
0,Argentina 0 - 0 Switzerland,7.81
19,Argentina 3 - 1 Switzerland,5.80
1,Argentina 0 - 1 Switzerland,5.15
24,Argentina 4 - 0 Switzerland,4.15
8,Argentina 1 - 2 Switzerland,3.21


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1131.5,0.80,905.2,120.0,785.2 - 1025.2,Baja
1,Pases completados totales,952.9,0.84,800.4,110.0,690.4 - 910.4,Baja
2,Tiros totales,25.3,0.82,20.7,5.0,15.7 - 25.7,Media-alta
3,Tiros al arco totales,8.4,0.82,6.9,3.0,3.9 - 9.9,Media
4,Corners totales,9.4,0.95,8.9,3.0,5.9 - 11.9,Media-alta
5,Saques de banda totales,37.8,1.00,37.8,6.0,31.8 - 43.8,Media
6,Faltas totales,26.4,0.73,19.3,5.0,14.3 - 24.3,Media
7,Amarillas totales,2.4,0.85,2.0,1.5,0.5 - 3.5,Media-alta


In [ ]:
goles_fra_esp, stats_fra_esp, totales_fra_esp, rangos_fra_esp, marcadores_fra_esp = reporte_total_partido("France", "Spain")

totales_fra_esp_ajustados = ajustar_totales_modelo(totales_fra_esp)
display(totales_fra_esp_ajustados)

REPORTE TOTAL: France vs Spain

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados France,0.85
1,Goles esperados Spain,1.52
2,Prob. gana France,21.1%
3,Prob. empate,26.0%
4,Prob. gana Spain,52.9%
5,Marcador más probable,0 - 1
6,Prob. marcador exacto,14.2%
7,Prob. France marca,57.3%
8,Prob. Spain marca,78.1%
9,Ambos marcan,44.7%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,France,605.8,512.6,84.8,18.3,6.6,6.2,20.9,11.5,1.5,50.4
1,Spain,595.2,501.1,84.1,18.1,7.7,7.0,20.5,10.8,1.7,49.6



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,France,pases,605.8,515.8 - 695.8
1,France,tiros,18.3,14.3 - 22.3
2,France,tiros_al_arco,6.6,4.6 - 8.6
3,France,corners,6.2,4.2 - 8.2
4,France,saques_banda,20.9,15.9 - 25.9
5,France,faltas_cometidas,11.5,7.5 - 15.5
6,France,tarjetas_amarillas,1.5,0 - 3.0
7,Spain,pases,595.2,505.2 - 685.2
8,Spain,tiros,18.1,14.1 - 22.1
9,Spain,tiros_al_arco,7.7,5.7 - 9.7



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
1,France 0 - 1 Spain,14.21
7,France 1 - 1 Spain,12.08
2,France 0 - 2 Spain,10.80
0,France 0 - 0 Spain,9.35
8,France 1 - 2 Spain,9.18
6,France 1 - 0 Spain,7.95
3,France 0 - 3 Spain,5.47
13,France 2 - 1 Spain,5.13
9,France 1 - 3 Spain,4.65
14,France 2 - 2 Spain,3.90


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1201.0,0.80,960.8,120.0,840.8 - 1080.8,Baja
1,Pases completados totales,1013.7,0.84,851.5,110.0,741.5 - 961.5,Baja
2,Tiros totales,36.4,0.82,29.8,5.0,24.8 - 34.8,Media-alta
3,Tiros al arco totales,14.3,0.82,11.7,3.0,8.7 - 14.7,Media
4,Corners totales,13.2,0.95,12.5,3.0,9.5 - 15.5,Media-alta
5,Saques de banda totales,41.4,1.00,41.4,6.0,35.4 - 47.4,Media
6,Faltas totales,22.3,0.73,16.3,5.0,11.3 - 21.3,Media
7,Amarillas totales,3.2,0.85,2.7,1.5,1.2 - 4.2,Media-alta


In [ ]:
goles_eng_arg, stats_eng_arg, totales_eng_arg, rangos_eng_arg, marcadores_eng_arg = reporte_total_partido("England", "Argentina")

totales_eng_arg_ajustados = ajustar_totales_modelo(totales_eng_arg)
display(totales_eng_arg_ajustados)

REPORTE TOTAL: England vs Argentina

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados England,0.98
1,Goles esperados Argentina,1.38
2,Prob. gana England,26.6%
3,Prob. empate,27.2%
4,Prob. gana Argentina,46.1%
5,Marcador más probable,0 - 1
6,Prob. marcador exacto,13.0%
7,Prob. England marca,62.5%
8,Prob. Argentina marca,74.8%
9,Ambos marcan,46.8%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,England,577.9,491.3,83.9,15.2,5.8,4.6,18.8,13.3,1.7,50.1
1,Argentina,575.8,495.4,85.6,12.9,4.2,5.2,18.3,12.9,1.1,49.9



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,England,pases,577.9,487.9 - 667.9
1,England,tiros,15.2,11.2 - 19.2
2,England,tiros_al_arco,5.8,3.8 - 7.8
3,England,corners,4.6,2.6 - 6.6
4,England,saques_banda,18.8,13.8 - 23.8
5,England,faltas_cometidas,13.3,9.3 - 17.3
6,England,tarjetas_amarillas,1.7,0.2 - 3.2
7,Argentina,pases,575.8,485.8 - 665.8
8,Argentina,tiros,12.9,8.9 - 16.9
9,Argentina,tiros_al_arco,4.2,2.2 - 6.2



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
1,England 0 - 1 Argentina,13.03
7,England 1 - 1 Argentina,12.77
0,England 0 - 0 Argentina,9.44
6,England 1 - 0 Argentina,9.25
2,England 0 - 2 Argentina,8.99
8,England 1 - 2 Argentina,8.81
13,England 2 - 1 Argentina,6.26
12,England 2 - 0 Argentina,4.53
14,England 2 - 2 Argentina,4.32
3,England 0 - 3 Argentina,4.14


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1153.7,0.80,923.0,120.0,803.0 - 1043.0,Baja
1,Pases completados totales,986.7,0.84,828.8,110.0,718.8 - 938.8,Baja
2,Tiros totales,28.1,0.82,23.0,5.0,18.0 - 28.0,Media-alta
3,Tiros al arco totales,10.0,0.82,8.2,3.0,5.2 - 11.2,Media
4,Corners totales,9.8,0.95,9.3,3.0,6.3 - 12.3,Media-alta
5,Saques de banda totales,37.1,1.00,37.1,6.0,31.1 - 43.1,Media
6,Faltas totales,26.2,0.73,19.1,5.0,14.1 - 24.1,Media
7,Amarillas totales,2.8,0.85,2.4,1.5,0.9 - 3.9,Media-alta


In [ ]:
goles_fra_eng, stats_fra_eng, totales_fra_eng, rangos_fra_eng, marcadores_fra_eng = reporte_total_partido("France", "England")

totales_fra_eng_ajustados = ajustar_totales_modelo(totales_fra_eng)
display(totales_fra_eng_ajustados)

REPORTE TOTAL: France vs England

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados France,1.23
1,Goles esperados England,1.05
2,Prob. gana France,40.2%
3,Prob. empate,28.4%
4,Prob. gana England,31.4%
5,Marcador más probable,1 - 1
6,Prob. marcador exacto,13.2%
7,Prob. France marca,70.8%
8,Prob. England marca,65.0%
9,Ambos marcan,46.0%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,France,605.8,512.6,84.8,18.3,6.6,6.2,20.9,11.5,1.5,51.2
1,England,577.7,491.0,83.9,15.1,5.8,4.7,18.8,13.5,1.8,48.8



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,France,pases,605.8,515.8 - 695.8
1,France,tiros,18.3,14.3 - 22.3
2,France,tiros_al_arco,6.6,4.6 - 8.6
3,France,corners,6.2,4.2 - 8.2
4,France,saques_banda,20.9,15.9 - 25.9
5,France,faltas_cometidas,11.5,7.5 - 15.5
6,France,tarjetas_amarillas,1.5,0 - 3.0
7,England,pases,577.7,487.7 - 667.7
8,England,tiros,15.1,11.1 - 19.1
9,England,tiros_al_arco,5.8,3.8 - 7.8



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
7,France 1 - 1 England,13.21
6,France 1 - 0 England,12.58
1,France 0 - 1 England,10.74
0,France 0 - 0 England,10.23
13,France 2 - 1 England,8.12
12,France 2 - 0 England,7.74
8,France 1 - 2 England,6.94
2,France 0 - 2 England,5.64
14,France 2 - 2 England,4.27
19,France 3 - 1 England,3.33


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1183.5,0.80,946.8,120.0,826.8 - 1066.8,Baja
1,Pases completados totales,1003.6,0.84,843.0,110.0,733.0 - 953.0,Baja
2,Tiros totales,33.4,0.82,27.4,5.0,22.4 - 32.4,Media-alta
3,Tiros al arco totales,12.4,0.82,10.2,3.0,7.2 - 13.2,Media
4,Corners totales,10.9,0.95,10.4,3.0,7.4 - 13.4,Media-alta
5,Saques de banda totales,39.7,1.00,39.7,6.0,33.7 - 45.7,Media
6,Faltas totales,25.0,0.73,18.2,5.0,13.2 - 23.2,Media
7,Amarillas totales,3.3,0.85,2.8,1.5,1.3 - 4.3,Media-alta


In [ ]:
goles_esp_arg, stats_esp_arg, totales_esp_arg, rangos_esp_arg, marcadores_esp_arg = reporte_total_partido("Spain", "Argentina")

totales_esp_arg_ajustados = ajustar_totales_modelo(totales_esp_arg)
display(totales_esp_arg_ajustados)

REPORTE TOTAL: Spain vs Argentina

1) GOLES Y RESULTADO


,Indicador,Valor
0,Goles esperados Spain,1.38
1,Goles esperados Argentina,0.92
2,Prob. gana Spain,47.7%
3,Prob. empate,27.4%
4,Prob. gana Argentina,24.9%
5,Marcador más probable,1 - 0
6,Prob. marcador exacto,13.8%
7,Prob. Spain marca,74.8%
8,Prob. Argentina marca,60.1%
9,Ambos marcan,45.0%



2) ESTADÍSTICAS POR EQUIPO


,equipo,pases,pases_completados,precision_pases_%,tiros,tiros_al_arco,corners,saques_banda,faltas_cometidas,tarjetas_amarillas,posesion_aprox_%
0,Spain,594.9,502.0,84.1,18.3,7.7,6.9,20.7,10.6,1.6,50.8
1,Argentina,575.8,495.4,85.6,12.9,4.2,5.2,18.3,12.9,1.1,49.2



3) TOTALES DEL PARTIDO


...


4) RANGOS APROXIMADOS


,equipo,estadística,estimado,rango_aprox
0,Spain,pases,594.9,504.9 - 684.9
1,Spain,tiros,18.3,14.3 - 22.3
2,Spain,tiros_al_arco,7.7,5.7 - 9.7
3,Spain,corners,6.9,4.9 - 8.9
4,Spain,saques_banda,20.7,15.7 - 25.7
5,Spain,faltas_cometidas,10.6,6.6 - 14.6
6,Spain,tarjetas_amarillas,1.6,0.1 - 3.1
7,Argentina,pases,575.8,485.8 - 665.8
8,Argentina,tiros,12.9,8.9 - 16.9
9,Argentina,tiros_al_arco,4.2,2.2 - 6.2



5) MARCADORES MÁS PROBABLES


,marcador,probabilidad_%
6,Spain 1 - 0 Argentina,13.84
7,Spain 1 - 1 Argentina,12.73
0,Spain 0 - 0 Argentina,10.03
12,Spain 2 - 0 Argentina,9.55
1,Spain 0 - 1 Argentina,9.22
13,Spain 2 - 1 Argentina,8.78
8,Spain 1 - 2 Argentina,5.86
18,Spain 3 - 0 Argentina,4.39
2,Spain 0 - 2 Argentina,4.24
14,Spain 2 - 2 Argentina,4.04


,Estadística total,Estimación,factor_ajuste,estimacion_ajustada,margen,rango_ajustado,confianza
0,Pases totales,1170.7,0.80,936.6,120.0,816.6 - 1056.6,Baja
1,Pases completados totales,997.4,0.84,837.8,110.0,727.8 - 947.8,Baja
2,Tiros totales,31.2,0.82,25.6,5.0,20.6 - 30.6,Media-alta
3,Tiros al arco totales,11.9,0.82,9.8,3.0,6.8 - 12.8,Media
4,Corners totales,12.1,0.95,11.5,3.0,8.5 - 14.5,Media-alta
5,Saques de banda totales,39.0,1.00,39.0,6.0,33.0 - 45.0,Media
6,Faltas totales,23.5,0.73,17.2,5.0,12.2 - 22.2,Media
7,Amarillas totales,2.7,0.85,2.3,1.5,0.8 - 3.8,Media-alta
